<a href="https://colab.research.google.com/github/solive-11/dissertation-weed-detection/blob/main/notebooks/04_baseline_object_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Purpose: Train and evaluate the first baseline weed object detector using the fixed leakage-aware train/val/test split created in Notebook 03.

from datetime import datetime
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/dissertation_weed_detection")

# Experiment metadata

NOTEBOOK_NAME = "04_baseline_object_detection"
EXPERIMENT_NAME = "baseline_yolo"

EXPERIMENT_ID = (
    f"{EXPERIMENT_NAME}_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

RANDOM_SEED = 42

# Important project directories

DATA_DIR = PROJECT_ROOT / "01_data"
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"

ENV_DIR = PROJECT_ROOT / "environment"
CONFIG_DIR = PROJECT_ROOT / "configs"
LOG_DIR = PROJECT_ROOT / "logs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"

# Experiment-specific directories
EXPERIMENT_LOG_DIR = LOG_DIR / EXPERIMENT_ID
EXPERIMENT_CHECKPOINT_DIR = CHECKPOINT_DIR / EXPERIMENT_ID
EXPERIMENT_RESULTS_DIR = RESULTS_DIR / EXPERIMENT_ID

# Create experiment directories

for directory in [
    EXPERIMENT_LOG_DIR,
    EXPERIMENT_CHECKPOINT_DIR,
    EXPERIMENT_RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Display metadata

print("=" * 70)
print("NOTEBOOK 04 — BASELINE OBJECT DETECTION")
print("=" * 70)

print(f"\nNotebook       : {NOTEBOOK_NAME}")
print(f"Experiment     : {EXPERIMENT_NAME}")
print(f"Experiment ID  : {EXPERIMENT_ID}")
print(f"Random seed    : {RANDOM_SEED}")

print("\nProject root:")
print(PROJECT_ROOT)

print("\nExperiment directories:")
print(f"  Logs         : {EXPERIMENT_LOG_DIR}")
print(f"  Checkpoints  : {EXPERIMENT_CHECKPOINT_DIR}")
print(f"  Results      : {EXPERIMENT_RESULTS_DIR}")

print("\n✓ Experiment workspace initialized")

NOTEBOOK 04 — BASELINE OBJECT DETECTION

Notebook       : 04_baseline_object_detection
Experiment     : baseline_yolo
Experiment ID  : baseline_yolo_20260909_065727
Random seed    : 42

Project root:
/content/drive/MyDrive/dissertation_weed_detection

Experiment directories:
  Logs         : /content/drive/MyDrive/dissertation_weed_detection/logs/baseline_yolo_20260909_065727
  Checkpoints  : /content/drive/MyDrive/dissertation_weed_detection/checkpoints/baseline_yolo_20260909_065727
  Results      : /content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_065727

✓ Experiment workspace initialized


In [4]:
# Verify known project structure

# Expected project structure
EXPECTED_PATHS = {
    "Project root": PROJECT_ROOT,

    "01_data": PROJECT_ROOT / "01_data",
    "Processed data": PROJECT_ROOT / "01_data" / "processed",
    "Splits": PROJECT_ROOT / "01_data" / "splits",

    "Environment": PROJECT_ROOT / "environment",
    "Configs": PROJECT_ROOT / "configs",
    "Logs": PROJECT_ROOT / "logs",
    "Checkpoints": PROJECT_ROOT / "checkpoints",
    "Results": PROJECT_ROOT / "results",
}

print("=" * 70)
print("PROJECT STRUCTURE VERIFICATION")
print("=" * 70)

for name, path in EXPECTED_PATHS.items():

    if path.exists():
        print(f"{name:<20}: ✓ exists")
    else:
        print(f"{name:<20}: ✗ MISSING")

print("\n" + "-" * 70)

# Inspect actual project root contents
print("\nPROJECT ROOT CONTENTS")
print("-" * 70)

if PROJECT_ROOT.exists():

    for item in sorted(PROJECT_ROOT.iterdir()):
        kind = "DIR " if item.is_dir() else "FILE"
        print(f"{kind}  {item.name}")

else:

    raise FileNotFoundError(
        f"Project root does not exist:\n{PROJECT_ROOT}"
    )

PROJECT STRUCTURE VERIFICATION
Project root        : ✓ exists
01_data             : ✓ exists
Processed data      : ✓ exists
Splits              : ✓ exists
Environment         : ✓ exists
Configs             : ✓ exists
Logs                : ✓ exists
Checkpoints         : ✓ exists
Results             : ✓ exists

----------------------------------------------------------------------

PROJECT ROOT CONTENTS
----------------------------------------------------------------------
DIR   01_data
DIR   02_notebooks
DIR   06_figures
DIR   checkpoints
DIR   configs
DIR   environment
DIR   logs
DIR   results


In [5]:
print("=" * 70)
print("VERIFYING EXISTING EXPERIMENT INPUTS")
print("=" * 70)

# Expected files from Notebook 3
manifest_candidates = [
    PROCESSED_DIR / "dataset_manifest_with_groups.csv",
    PROCESSED_DIR / "dataset_manifest.csv",
]

print("\nMANIFEST FILES")
print("-" * 70)

manifest_found = None

for path in manifest_candidates:
    if path.exists():
        print(f"✓ {path}")
        manifest_found = path
    else:
        print(f"✗ {path}")

if manifest_found is None:
    raise FileNotFoundError(
        "The dataset manifest from Notebook 3 could not be found."
    )

# List relevant processed files
print("\nPROCESSED DIRECTORY")
print("-" * 70)

for path in sorted(PROCESSED_DIR.iterdir()):
    if path.is_file():
        print(f"FILE : {path.name}")
    elif path.is_dir():
        print(f"DIR  : {path.name}")

# List split directory
print("\nSPLITS DIRECTORY")
print("-" * 70)

for path in sorted(SPLITS_DIR.iterdir()):
    if path.is_file():
        print(f"FILE : {path.name}")
    elif path.is_dir():
        print(f"DIR  : {path.name}")

print("\n" + "=" * 70)
print("✓ EXISTING EXPERIMENT INPUTS VERIFIED")
print("=" * 70)

print(f"\nManifest selected:")
print(manifest_found)

VERIFYING EXISTING EXPERIMENT INPUTS

MANIFEST FILES
----------------------------------------------------------------------
✓ /content/drive/MyDrive/dissertation_weed_detection/01_data/processed/dataset_manifest_with_groups.csv
✓ /content/drive/MyDrive/dissertation_weed_detection/01_data/processed/dataset_manifest.csv

PROCESSED DIRECTORY
----------------------------------------------------------------------
FILE : dataset_manifest.csv
FILE : dataset_manifest_with_groups.csv
FILE : dataset_summary.json
DIR  : quality_checks

SPLITS DIRECTORY
----------------------------------------------------------------------

✓ EXISTING EXPERIMENT INPUTS VERIFIED

Manifest selected:
/content/drive/MyDrive/dissertation_weed_detection/01_data/processed/dataset_manifest.csv


In [7]:
# ============================================================
# Cell 4A: Locate the saved train/val/test split artifact
# ============================================================

print("=" * 70)
print("SEARCHING FOR SAVED SPLIT ARTIFACTS")
print("=" * 70)

# Search the project's data/split/config directories
search_dirs = [
    DATA_DIR,
    PROCESSED_DIR,
    SPLITS_DIR,
    CONFIG_DIR,
    RESULTS_DIR,
    LOG_DIR,
]

patterns = [
    "*split*",
    "*train*",
    "*val*",
    "*test*",
    "*.csv",
    "*.json",
    "*.yaml",
    "*.yml",
]

found = set()

for directory in search_dirs:

    if not directory.exists():
        continue

    for pattern in patterns:

        for path in directory.rglob(pattern):

            if path.is_file():
                found.add(path)

print(f"\nFiles found: {len(found)}")

for path in sorted(found):
    print(path)

print("\n" + "=" * 70)
print("SEARCH COMPLETE")
print("=" * 70)

Streaming output truncated to the last 5000 lines.
/content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json/11085p0y44nx242524_954.json
/content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json/11085p2gnmjx232555_172.json
/content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json/11085p7044ob442553_145.json
/content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json/11085p7vgv0n502500_142.json
/content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json/11085p89m2pe422408_495.json
/content/

In [11]:
# ============================================================
# Cell 5: EXACT reconstruction of Notebook 3 split
# ============================================================

from collections import defaultdict, Counter
import numpy as np
import pandas as pd

print("=" * 70)
print("EXACT RECONSTRUCTION OF NOTEBOOK 3 SPLIT")
print("=" * 70)

# ------------------------------------------------------------
# Configuration — EXACTLY as Notebook 3
# ------------------------------------------------------------

SPLIT_RATIOS = {
    "train": 0.80,
    "val": 0.10,
    "test": 0.10
}

SEED = 42
GROUP_COLUMN = "near_duplicate_group"

rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# Load the exact manifest
# ------------------------------------------------------------

MANIFEST_PATH = (
    PROCESSED_DIR /
    "dataset_manifest_with_groups.csv"
)

df = pd.read_csv(MANIFEST_PATH)

print(f"\nManifest       : {MANIFEST_PATH}")
print(f"Images         : {len(df):,}")
print(f"Unique groups  : {df[GROUP_COLUMN].nunique():,}")

# ------------------------------------------------------------
# IMPORTANT:
# Notebook 3 already computed image_classes and group_classes.
#
# Reconstruct those exactly from the annotations.
# ------------------------------------------------------------

def get_classes(annotation_path):

    import json

    with open(annotation_path, "r") as f:
        annotation = json.load(f)

    if isinstance(annotation, list):
        objects = annotation

    elif isinstance(annotation, dict):
        objects = annotation.get("objects", [])

    else:
        objects = []

    classes = set()

    for obj in objects:

        if not isinstance(obj, dict):
            continue

        class_id = obj.get(
            "class_id",
            obj.get("class")
        )

        if class_id is None:
            continue

        try:
            classes.add(int(class_id))
        except (ValueError, TypeError):
            continue

    return classes


print("\nReading annotation classes...")

image_classes = {}

for _, row in df.iterrows():

    image_id = row["image_id"]

    image_classes[image_id] = get_classes(
        row["annotation_path"]
    )

print("✓ Image class information reconstructed")

# ------------------------------------------------------------
# Reconstruct group_classes
# ------------------------------------------------------------

group_classes = {}

for group_id, indices in df.groupby(
    GROUP_COLUMN
).groups.items():

    classes = set()

    for idx in indices:

        image_id = df.loc[idx, "image_id"]

        classes.update(
            image_classes[image_id]
        )

    group_classes[group_id] = classes

print(
    f"✓ Group class information reconstructed "
    f"for {len(group_classes):,} groups"
)

# ------------------------------------------------------------
# EXACT NOTEBOOK 3 CODE
# ------------------------------------------------------------

unique_groups = list(
    df[GROUP_COLUMN].unique()
)

n_groups = len(unique_groups)

target_groups = {
    split: round(n_groups * ratio)
    for split, ratio in SPLIT_RATIOS.items()
}

difference = (
    n_groups -
    sum(target_groups.values())
)

target_groups["train"] += difference

print("\nTARGET GROUPS")
print("-" * 70)

for split, count in target_groups.items():
    print(f"{split:<10}: {count:,}")

# ------------------------------------------------------------
# Build class -> groups mapping
# ------------------------------------------------------------

class_to_groups = defaultdict(set)

for group_id, classes in group_classes.items():

    for class_id in classes:

        class_to_groups[class_id].add(
            group_id
        )

# ------------------------------------------------------------
# EXACT GROUP ORDERING FROM NOTEBOOK 3
# ------------------------------------------------------------

group_list = list(unique_groups)

rng.shuffle(group_list)

group_list.sort(
    key=lambda g: (
        -sum(
            1 / max(
                len(class_to_groups[c]),
                1
            )
            for c in group_classes[g]
        ),
        -len(group_classes[g])
    )
)

# ------------------------------------------------------------
# Initialize assignment structures
# ------------------------------------------------------------

assignments = {}

split_group_counts = {
    "train": 0,
    "val": 0,
    "test": 0
}

class_group_targets = {}

# NOTEBOOK 3 uses class_names
# Reconstruct it from observed classes.

class_names = sorted(
    class_to_groups.keys()
)

for class_id in class_names:

    total = len(
        class_to_groups[class_id]
    )

    class_group_targets[class_id] = {
        split: total * ratio
        for split, ratio in SPLIT_RATIOS.items()
    }

class_group_current = {
    split: Counter()
    for split in SPLIT_RATIOS
}

# ------------------------------------------------------------
# EXACT NOTEBOOK 3 ASSIGNMENT LOOP
# ------------------------------------------------------------

for group_id in group_list:

    classes = group_classes[group_id]

    available_splits = [
        split
        for split in SPLIT_RATIOS
        if (
            split_group_counts[split]
            < target_groups[split]
        )
    ]

    if not available_splits:

        raise RuntimeError(
            "No available split while groups remain."
        )

    scores = {}

    for split in available_splits:

        score = 0.0

        for class_id in classes:

            target = (
                class_group_targets[
                    class_id
                ][split]
            )

            current = (
                class_group_current[
                    split
                ][class_id]
            )

            if target > 0:

                score += max(
                    0,
                    target - current
                ) / target

        # Exact Notebook 3 capacity term
        capacity_ratio = (
            target_groups[split]
            - split_group_counts[split]
        ) / target_groups[split]

        score += 0.01 * capacity_ratio

        scores[split] = score

    # Exact deterministic tie breaking
    best_score = max(
        scores.values()
    )

    best_splits = [
        split
        for split, score in scores.items()
        if np.isclose(
            score,
            best_score
        )
    ]

    chosen_split = sorted(
        best_splits
    )[0]

    assignments[group_id] = chosen_split

    split_group_counts[
        chosen_split
    ] += 1

    for class_id in classes:

        class_group_current[
            chosen_split
        ][class_id] += 1

# ------------------------------------------------------------
# Add reconstructed split
# ------------------------------------------------------------

df["split"] = (
    df[GROUP_COLUMN]
    .map(assignments)
)

if df["split"].isna().any():

    raise RuntimeError(
        "Some images were not assigned to a split."
    )

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RECONSTRUCTED SPLIT")
print("=" * 70)

for split in ["train", "val", "test"]:

    subset = df[
        df["split"] == split
    ]

    print(
        f"{split:<10}: "
        f"{len(subset):,} images | "
        f"{subset[GROUP_COLUMN].nunique():,} groups"
    )

# ------------------------------------------------------------
# Exact expected values from Notebook 3
# ------------------------------------------------------------

expected = {
    "train": {
        "images": 5326,
        "groups": 5151
    },
    "val": {
        "images": 668,
        "groups": 644
    },
    "test": {
        "images": 662,
        "groups": 644
    }
}

print("\n" + "=" * 70)
print("COMPARISON WITH NOTEBOOK 3")
print("=" * 70)

exact_counts = True

for split in ["train", "val", "test"]:

    subset = df[
        df["split"] == split
    ]

    actual_images = len(subset)

    actual_groups = (
        subset[GROUP_COLUMN]
        .nunique()
    )

    expected_images = (
        expected[split]["images"]
    )

    expected_groups = (
        expected[split]["groups"]
    )

    image_ok = (
        actual_images ==
        expected_images
    )

    group_ok = (
        actual_groups ==
        expected_groups
    )

    print(
        f"{split:<6}: "
        f"Images {actual_images:,} / "
        f"{expected_images:,} "
        f"{'✓' if image_ok else '✗'} | "
        f"Groups {actual_groups:,} / "
        f"{expected_groups:,} "
        f"{'✓' if group_ok else '✗'}"
    )

    if not (image_ok and group_ok):
        exact_counts = False

print("\n" + "=" * 70)

if exact_counts:

    print(
        "✓ SPLIT COUNTS EXACTLY MATCH NOTEBOOK 3"
    )

else:

    print(
        "✗ SPLIT COUNTS DO NOT MATCH NOTEBOOK 3"
    )

    raise RuntimeError(
        "Reconstruction failed. "
        "DO NOT save or use this split."
    )

EXACT RECONSTRUCTION OF NOTEBOOK 3 SPLIT

Manifest       : /content/drive/MyDrive/dissertation_weed_detection/01_data/processed/dataset_manifest_with_groups.csv
Images         : 6,656
Unique groups  : 6,439

Reading annotation classes...
✓ Image class information reconstructed
✓ Group class information reconstructed for 6,439 groups

TARGET GROUPS
----------------------------------------------------------------------
train     : 5,151
val       : 644
test      : 644

RECONSTRUCTED SPLIT
train     : 5,326 images | 5,151 groups
val       : 668 images | 644 groups
test      : 662 images | 644 groups

COMPARISON WITH NOTEBOOK 3
train : Images 5,326 / 5,326 ✓ | Groups 5,151 / 5,151 ✓
val   : Images 668 / 668 ✓ | Groups 644 / 644 ✓
test  : Images 662 / 662 ✓ | Groups 644 / 644 ✓

✓ SPLIT COUNTS EXACTLY MATCH NOTEBOOK 3


In [12]:
# ============================================================
# Cell 6: Persist the verified experiment split
# ============================================================

import json
from datetime import datetime

print("=" * 70)
print("PERSISTING VERIFIED EXPERIMENT SPLIT")
print("=" * 70)

# ------------------------------------------------------------
# Create split directory if necessary
# ------------------------------------------------------------

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Verify once more before writing
# ------------------------------------------------------------

expected = {
    "train": {"images": 5326, "groups": 5151},
    "val":   {"images": 668,  "groups": 644},
    "test":  {"images": 662,  "groups": 644},
}

for split_name, expected_values in expected.items():

    subset = df[df["split"] == split_name]

    assert len(subset) == expected_values["images"], (
        f"{split_name}: image count mismatch"
    )

    assert subset[GROUP_COLUMN].nunique() == expected_values["groups"], (
        f"{split_name}: group count mismatch"
    )

# ------------------------------------------------------------
# Verify no group crosses splits
# ------------------------------------------------------------

group_split_counts = (
    df.groupby(GROUP_COLUMN)["split"]
      .nunique()
)

leaking_groups = group_split_counts[
    group_split_counts > 1
]

assert len(leaking_groups) == 0, (
    f"Found {len(leaking_groups)} groups crossing splits"
)

print("\n✓ Image counts verified")
print("✓ Group counts verified")
print("✓ No group leakage")

# ------------------------------------------------------------
# Save complete split manifest
# ------------------------------------------------------------

SPLIT_MANIFEST_PATH = (
    SPLITS_DIR / "split_manifest.csv"
)

df.to_csv(
    SPLIT_MANIFEST_PATH,
    index=False
)

print(
    f"\n✓ Saved complete split manifest:\n"
    f"  {SPLIT_MANIFEST_PATH}"
)

# ------------------------------------------------------------
# Save individual split files
# ------------------------------------------------------------

split_files = {}

for split_name in ["train", "val", "test"]:

    split_df = df[
        df["split"] == split_name
    ].copy()

    path = (
        SPLITS_DIR /
        f"{split_name}.csv"
    )

    split_df.to_csv(
        path,
        index=False
    )

    split_files[split_name] = str(path)

    print(
        f"✓ {split_name:<5} → "
        f"{len(split_df):,} images"
    )

# ------------------------------------------------------------
# Save split configuration
# ------------------------------------------------------------

config = {
    "dataset": "MH-Weed16",
    "task": "weed_object_detection",

    "manifest_source": str(MANIFEST_PATH),

    "group_column": GROUP_COLUMN,

    "seed": SEED,

    "split_ratios": {
        "train": TRAIN_RATIO,
        "val": VAL_RATIO,
        "test": TEST_RATIO
    },

    "target_groups": {
        "train": 5151,
        "val": 644,
        "test": 644
    },

    "actual_groups": {
        "train": int(
            df[df["split"] == "train"][GROUP_COLUMN].nunique()
        ),
        "val": int(
            df[df["split"] == "val"][GROUP_COLUMN].nunique()
        ),
        "test": int(
            df[df["split"] == "test"][GROUP_COLUMN].nunique()
        )
    },

    "actual_images": {
        "train": int(
            (df["split"] == "train").sum()
        ),
        "val": int(
            (df["split"] == "val").sum()
        ),
        "test": int(
            (df["split"] == "test").sum()
        )
    },

    "near_duplicate_leakage_groups": 0,

    "source_notebook": (
        "03_dataset_split_and_experiment_setup.ipynb"
    ),

    "created_at": datetime.now().isoformat()
}

CONFIG_PATH = (
    CONFIG_DIR /
    "baseline_split_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(
        config,
        f,
        indent=2
    )

print(
    f"\n✓ Saved split configuration:\n"
    f"  {CONFIG_PATH}"
)

# ------------------------------------------------------------
# Save verification log
# ------------------------------------------------------------

LOG_PATH = (
    LOG_DIR /
    "split_persistence_verification.txt"
)

with open(LOG_PATH, "w") as f:

    f.write(
        "MH-Weed16 EXPERIMENT SPLIT VERIFICATION\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write(
        "Source notebook: "
        "03_dataset_split_and_experiment_setup.ipynb\n"
    )
    f.write(f"Seed: {SEED}\n")
    f.write(
        "Split ratio: 80/10/10\n\n"
    )

    f.write("FINAL SPLIT\n")
    f.write("-" * 70 + "\n")

    for split_name in ["train", "val", "test"]:

        subset = df[
            df["split"] == split_name
        ]

        f.write(
            f"{split_name}: "
            f"{len(subset)} images, "
            f"{subset[GROUP_COLUMN].nunique()} groups\n"
        )

    f.write("\n")
    f.write(
        "Near-duplicate groups crossing splits: 0\n"
    )
    f.write(
        "Status: VERIFIED\n"
    )

print(
    f"\n✓ Saved verification log:\n"
    f"  {LOG_PATH}"
)

print("\n" + "=" * 70)
print("✓ VERIFIED SPLIT PERMANENTLY SAVED")
print("=" * 70)

PERSISTING VERIFIED EXPERIMENT SPLIT

✓ Image counts verified
✓ Group counts verified
✓ No group leakage

✓ Saved complete split manifest:
  /content/drive/MyDrive/dissertation_weed_detection/01_data/splits/split_manifest.csv
✓ train → 5,326 images
✓ val   → 668 images
✓ test  → 662 images

✓ Saved split configuration:
  /content/drive/MyDrive/dissertation_weed_detection/configs/baseline_split_config.json

✓ Saved verification log:
  /content/drive/MyDrive/dissertation_weed_detection/logs/split_persistence_verification.txt

✓ VERIFIED SPLIT PERMANENTLY SAVED


In [13]:
# ============================================================
# Cell 7: Verify persisted split artifacts

print("=" * 70)
print("VERIFYING PERSISTED EXPERIMENT SPLIT")
print("=" * 70)

split_paths = {
    "train": SPLITS_DIR / "train.csv",
    "val":   SPLITS_DIR / "val.csv",
    "test":  SPLITS_DIR / "test.csv",
}

expected = {
    "train": {"images": 5326, "groups": 5151},
    "val":   {"images": 668,  "groups": 644},
    "test":  {"images": 662,  "groups": 644},
}

for split_name, path in split_paths.items():

    assert path.exists(), (
        f"Missing split file: {path}"
    )

    split_df = pd.read_csv(path)

    image_count = len(split_df)
    group_count = split_df[
        GROUP_COLUMN
    ].nunique()

    print(
        f"\n{split_name.upper()}"
    )
    print("-" * 70)
    print(f"File   : {path}")
    print(f"Images : {image_count:,}")
    print(f"Groups : {group_count:,}")

    assert image_count == expected[split_name]["images"]
    assert group_count == expected[split_name]["groups"]

print("\n" + "=" * 70)
print("✓ ALL PERSISTED SPLITS VERIFIED")
print("=" * 70)

VERIFYING PERSISTED EXPERIMENT SPLIT

TRAIN
----------------------------------------------------------------------
File   : /content/drive/MyDrive/dissertation_weed_detection/01_data/splits/train.csv
Images : 5,326
Groups : 5,151

VAL
----------------------------------------------------------------------
File   : /content/drive/MyDrive/dissertation_weed_detection/01_data/splits/val.csv
Images : 668
Groups : 644

TEST
----------------------------------------------------------------------
File   : /content/drive/MyDrive/dissertation_weed_detection/01_data/splits/test.csv
Images : 662
Groups : 644

✓ ALL PERSISTED SPLITS VERIFIED


In [15]:
# ============================================================
# Cell 8: Create YOLO experiment dataset structure

print("=" * 70)
print("CREATING YOLO EXPERIMENT DATASET STRUCTURE")
print("=" * 70)

# ------------------------------------------------------------
# Explicit project paths

DATA_DIR = PROJECT_ROOT / "01_data"
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"

CONFIG_DIR = PROJECT_ROOT / "configs"
LOG_DIR = PROJECT_ROOT / "logs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"

# ------------------------------------------------------------
# Verify project root
# ------------------------------------------------------------

assert PROJECT_ROOT.exists(), (
    f"Project root not found: {PROJECT_ROOT}"
)

print(f"\nProject root : {PROJECT_ROOT}")

# ------------------------------------------------------------
# YOLO dataset root
# ------------------------------------------------------------

YOLO_DATASET_DIR = (
    DATA_DIR / "yolo_baseline"
)

# ------------------------------------------------------------
# Standard YOLO directory structure
# ------------------------------------------------------------

YOLO_DIRS = {
    "train_images": YOLO_DATASET_DIR / "images" / "train",
    "val_images":   YOLO_DATASET_DIR / "images" / "val",
    "test_images":  YOLO_DATASET_DIR / "images" / "test",

    "train_labels": YOLO_DATASET_DIR / "labels" / "train",
    "val_labels":   YOLO_DATASET_DIR / "labels" / "val",
    "test_labels":  YOLO_DATASET_DIR / "labels" / "test",
}

# ------------------------------------------------------------
# Create directories
# ------------------------------------------------------------

for path in YOLO_DIRS.values():

    path.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\nYOLO DATASET ROOT")
print("-" * 70)
print(YOLO_DATASET_DIR)

print("\nDIRECTORIES")
print("-" * 70)

for name, path in YOLO_DIRS.items():

    status = "✓ exists" if path.exists() else "✗ missing"

    print(
        f"{name:<15}: {status}"
    )

assert all(
    path.exists()
    for path in YOLO_DIRS.values()
)

print("\n" + "=" * 70)
print("✓ YOLO DATASET STRUCTURE CREATED")
print("=" * 70)

CREATING YOLO EXPERIMENT DATASET STRUCTURE

Project root : /content/drive/MyDrive/dissertation_weed_detection

YOLO DATASET ROOT
----------------------------------------------------------------------
/content/drive/MyDrive/dissertation_weed_detection/01_data/yolo_baseline

DIRECTORIES
----------------------------------------------------------------------
train_images   : ✓ exists
val_images     : ✓ exists
test_images    : ✓ exists
train_labels   : ✓ exists
val_labels     : ✓ exists
test_labels    : ✓ exists

✓ YOLO DATASET STRUCTURE CREATED


In [16]:
# ============================================================
# Verify source image and annotation availability

print("=" * 70)
print("VERIFYING SOURCE IMAGE / ANNOTATION PAIRS")
print("=" * 70)

# ------------------------------------------------------------
# Load the FIXED split manifest
# ------------------------------------------------------------

SPLIT_MANIFEST_PATH = (
    SPLITS_DIR / "split_manifest.csv"
)

assert SPLIT_MANIFEST_PATH.exists(), (
    f"Split manifest not found:\n{SPLIT_MANIFEST_PATH}"
)

split_df = pd.read_csv(
    SPLIT_MANIFEST_PATH
)

print(f"\nManifest : {SPLIT_MANIFEST_PATH}")
print(f"Rows     : {len(split_df):,}")

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "image_id",
    "image_path",
    "annotation_path",
    "image_filename",
    "near_duplicate_group",
    "split"
]

missing_columns = [
    c for c in required_columns
    if c not in split_df.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

print("\n✓ Required columns present")

# ------------------------------------------------------------
# Check source files
# ------------------------------------------------------------

image_exists = []
annotation_exists = []

for _, row in split_df.iterrows():

    image_path = Path(
        str(row["image_path"])
    )

    annotation_path = Path(
        str(row["annotation_path"])
    )

    image_exists.append(
        image_path.exists()
    )

    annotation_exists.append(
        annotation_path.exists()
    )

split_df["_image_exists"] = image_exists
split_df["_annotation_exists"] = annotation_exists

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

missing_images = (
    (~split_df["_image_exists"])
    .sum()
)

missing_annotations = (
    (~split_df["_annotation_exists"])
    .sum()
)

print("\nSOURCE FILE CHECK")
print("-" * 70)

print(
    f"Images found       : "
    f"{split_df['_image_exists'].sum():,} / {len(split_df):,}"
)

print(
    f"Images missing     : "
    f"{missing_images:,}"
)

print(
    f"Annotations found  : "
    f"{split_df['_annotation_exists'].sum():,} / {len(split_df):,}"
)

print(
    f"Annotations missing: "
    f"{missing_annotations:,}"
)

# ------------------------------------------------------------
# Show missing examples if any
# ------------------------------------------------------------

if missing_images > 0:

    print("\nFIRST MISSING IMAGES")
    print("-" * 70)

    print(
        split_df.loc[
            ~split_df["_image_exists"],
            "image_path"
        ].head(10).to_string(index=False)
    )

if missing_annotations > 0:

    print("\nFIRST MISSING ANNOTATIONS")
    print("-" * 70)

    print(
        split_df.loc[
            ~split_df["_annotation_exists"],
            "annotation_path"
        ].head(10).to_string(index=False)
    )

# ------------------------------------------------------------
# Final integrity assertion
# ------------------------------------------------------------

assert missing_images == 0, (
    "Some source images are missing."
)

assert missing_annotations == 0, (
    "Some source annotations are missing."
)

print("\n" + "=" * 70)
print("✓ ALL SOURCE IMAGE / ANNOTATION PAIRS ARE AVAILABLE")
print("=" * 70)

VERIFYING SOURCE IMAGE / ANNOTATION PAIRS

Manifest : /content/drive/MyDrive/dissertation_weed_detection/01_data/splits/split_manifest.csv
Rows     : 6,656

✓ Required columns present

SOURCE FILE CHECK
----------------------------------------------------------------------
Images found       : 6,656 / 6,656
Images missing     : 0
Annotations found  : 6,656 / 6,656
Annotations missing: 0

✓ ALL SOURCE IMAGE / ANNOTATION PAIRS ARE AVAILABLE


In [17]:
# ============================================================
# Inspect representative JSON annotations

print("=" * 70)
print("INSPECTING REPRESENTATIVE JSON ANNOTATIONS")
print("=" * 70)

# Select a few representative images
sample_rows = split_df.sample(
    n=min(5, len(split_df)),
    random_state=SEED
)

for i, (_, row) in enumerate(sample_rows.iterrows(), start=1):

    annotation_path = Path(
        str(row["annotation_path"])
    )

    print("\n" + "-" * 70)
    print(f"Sample {i}")
    print(f"Image      : {row['image_filename']}")
    print(f"Annotation : {annotation_path}")

    with open(annotation_path, "r") as f:
        annotation = json.load(f)

    print(f"JSON type  : {type(annotation).__name__}")

    if isinstance(annotation, dict):

        print(
            "Top-level keys:"
        )

        print(
            list(annotation.keys())
        )

        for key, value in annotation.items():

            if isinstance(value, list):

                print(
                    f"  {key}: list "
                    f"(length={len(value)})"
                )

            else:

                print(
                    f"  {key}: "
                    f"{type(value).__name__}"
                )

    elif isinstance(annotation, list):

        print(
            f"List length: {len(annotation)}"
        )

        if len(annotation) > 0:

            print(
                "First element type:"
                f" {type(annotation[0]).__name__}"
            )

            if isinstance(annotation[0], dict):

                print(
                    "First element keys:"
                )

                print(
                    list(annotation[0].keys())
                )

                print(
                    "\nFirst element:"
                )

                print(
                    json.dumps(
                        annotation[0],
                        indent=2
                    )[:2000]
                )

            else:

                print(
                    "\nFirst element:"
                )

                print(
                    repr(annotation[0])
                )

print("\n" + "=" * 70)
print("✓ ANNOTATION STRUCTURE INSPECTION COMPLETE")
print("=" * 70)

INSPECTING REPRESENTATIVE JSON ANNOTATIONS

----------------------------------------------------------------------
Sample 1
Image      : 1508x03esk6l482448_128.jpeg
Annotation : /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json/1508x03esk6l482448_128.json
JSON type  : list
List length: 11
First element type: dict
First element keys:
['class_id', 'x_center', 'y_center', 'width', 'height']

First element:
{
  "class_id": 2,
  "x_center": 0.7359375,
  "y_center": 0.11898148148148148,
  "width": 0.029166666666666667,
  "height": 0.04537037037037037
}

----------------------------------------------------------------------
Sample 2
Image      : 100836ps22wz012401_157.jpeg
Annotation : /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json/100836ps22wz012401_157.json
JSON type  : l

In [19]:
# ============================================================
# Cell 11: Full annotation validation

print("=" * 70)
print("FULL ANNOTATION VALIDATION")
print("=" * 70)

EXPECTED_CLASSES = set(range(15))

required_fields = {
    "class_id",
    "x_center",
    "y_center",
    "width",
    "height"
}

stats = {
    "images_checked": 0,
    "total_boxes": 0,
    "invalid_json": 0,
    "invalid_structure": 0,
    "invalid_fields": 0,
    "invalid_class": 0,
    "invalid_coordinates": 0,
    "invalid_dimensions": 0,
}

problems = []

# ------------------------------------------------------------
# Validate every annotation
# ------------------------------------------------------------

for idx, row in split_df.iterrows():

    annotation_path = Path(
        str(row["annotation_path"])
    )

    image_name = str(
        row["image_filename"]
    )

    stats["images_checked"] += 1

    try:

        with open(
            annotation_path,
            "r"
        ) as f:

            annotation = json.load(f)

    except Exception as e:

        stats["invalid_json"] += 1

        problems.append(
            (
                image_name,
                "JSON_READ_ERROR",
                str(e)
            )
        )

        continue

    # --------------------------------------------------------
    # Expected structure: list
    # --------------------------------------------------------

    if not isinstance(annotation, list):

        stats["invalid_structure"] += 1

        problems.append(
            (
                image_name,
                "NOT_A_LIST",
                type(annotation).__name__
            )
        )

        continue

    # --------------------------------------------------------
    # Validate each bounding box
    # --------------------------------------------------------

    for box_index, box in enumerate(annotation):

        stats["total_boxes"] += 1

        if not isinstance(box, dict):

            stats["invalid_structure"] += 1

            problems.append(
                (
                    image_name,
                    f"BOX_{box_index}_NOT_DICT",
                    type(box).__name__
                )
            )

            continue

        # ----------------------------------------------------
        # Required fields
        # ----------------------------------------------------

        missing = required_fields - set(box.keys())

        if missing:

            stats["invalid_fields"] += 1

            problems.append(
                (
                    image_name,
                    f"BOX_{box_index}_MISSING_FIELDS",
                    str(sorted(missing))
                )
            )

            continue

        # ----------------------------------------------------
        # Class ID
        # ----------------------------------------------------

        try:

            class_id = int(
                box["class_id"]
            )

        except Exception:

            stats["invalid_class"] += 1

            problems.append(
                (
                    image_name,
                    f"BOX_{box_index}_INVALID_CLASS",
                    repr(box["class_id"])
                )
            )

            continue

        if class_id not in EXPECTED_CLASSES:

            stats["invalid_class"] += 1

            problems.append(
                (
                    image_name,
                    f"BOX_{box_index}_CLASS_OUT_OF_RANGE",
                    str(class_id)
                )
            )

        # ----------------------------------------------------
        # Coordinates
        # ----------------------------------------------------

        try:

            x = float(box["x_center"])
            y = float(box["y_center"])
            w = float(box["width"])
            h = float(box["height"])

        except Exception:

            stats["invalid_coordinates"] += 1

            problems.append(
                (
                    image_name,
                    f"BOX_{box_index}_NON_NUMERIC",
                    str(box)
                )
            )

            continue

        # ----------------------------------------------------
        # Dimensions must be positive
        # ----------------------------------------------------

        if w <= 0 or h <= 0:

            stats["invalid_dimensions"] += 1

            problems.append(
                (
                    image_name,
                    f"BOX_{box_index}_NON_POSITIVE_SIZE",
                    f"w={w}, h={h}"
                )
            )

        # ----------------------------------------------------
        # Normalized YOLO coordinates
        # ----------------------------------------------------

        if not (
            0 <= x <= 1 and
            0 <= y <= 1 and
            0 < w <= 1 and
            0 < h <= 1
        ):

            stats["invalid_coordinates"] += 1

            problems.append(
                (
                    image_name,
                    f"BOX_{box_index}_OUT_OF_RANGE",
                    f"x={x}, y={y}, w={w}, h={h}"
                )
            )

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nVALIDATION SUMMARY")
print("-" * 70)

for key, value in stats.items():

    print(
        f"{key:<25}: {value:,}"
    )

# ------------------------------------------------------------
# Problem summary
# ------------------------------------------------------------

print("\nPROBLEMS FOUND")
print("-" * 70)

total_problems = sum(
    value
    for key, value in stats.items()
    if key.startswith("invalid_")
)

print(
    f"Total validation problems : "
    f"{total_problems:,}"
)

if problems:

    print("\nFIRST 10 PROBLEMS")
    print("-" * 70)

    for problem in problems[:10]:

        print(problem)

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert stats["images_checked"] == 6656

assert stats["invalid_json"] == 0
assert stats["invalid_structure"] == 0
assert stats["invalid_fields"] == 0
assert stats["invalid_class"] == 0
assert stats["invalid_coordinates"] == 0
assert stats["invalid_dimensions"] == 0

assert stats["total_boxes"] == 62052

print("\n" + "=" * 70)
print("✓ ALL 6,656 ANNOTATIONS VALID")
print("✓ ALL 62,052 BOUNDING BOXES VALID")
print("✓ CLASS IDs LIMITED TO 0–14")
print("✓ COORDINATES ARE VALID NORMALIZED YOLO VALUES")
print("=" * 70)

FULL ANNOTATION VALIDATION

VALIDATION SUMMARY
----------------------------------------------------------------------
images_checked           : 6,656
total_boxes              : 62,052
invalid_json             : 0
invalid_structure        : 0
invalid_fields           : 0
invalid_class            : 0
invalid_coordinates      : 12
invalid_dimensions       : 12

PROBLEMS FOUND
----------------------------------------------------------------------
Total validation problems : 24

FIRST 10 PROBLEMS
----------------------------------------------------------------------
('0908tis3cgp3452411_120.jpeg', 'BOX_1_NON_POSITIVE_SIZE', 'w=0.22552083333333334, h=0.0')
('0908tis3cgp3452411_120.jpeg', 'BOX_1_OUT_OF_RANGE', 'x=0.8861979166666667, y=0.5148148148148148, w=0.22552083333333334, h=0.0')
('1108u04wol98472626_117.jpeg', 'BOX_2_NON_POSITIVE_SIZE', 'w=0.0, h=0.028703703703703703')
('1108u04wol98472626_117.jpeg', 'BOX_2_OUT_OF_RANGE', 'x=0.0, y=0.16712962962962963, w=0.0, h=0.028703703703703703')
(

AssertionError: 

In [20]:
# ============================================================
# Cell 12: Audit malformed bounding boxes

print("=" * 70)
print("AUDITING MALFORMED BOUNDING BOXES")
print("=" * 70)

bad_boxes = []

for _, row in split_df.iterrows():

    annotation_path = Path(
        str(row["annotation_path"])
    )

    image_name = str(
        row["image_filename"]
    )

    with open(annotation_path, "r") as f:
        annotation = json.load(f)

    for box_index, box in enumerate(annotation):

        class_id = int(box["class_id"])

        x = float(box["x_center"])
        y = float(box["y_center"])
        w = float(box["width"])
        h = float(box["height"])

        if w <= 0 or h <= 0:

            bad_boxes.append({
                "image_id": row["image_id"],
                "image_filename": image_name,
                "annotation_path": str(annotation_path),
                "split": row["split"],
                "group": row["near_duplicate_group"],
                "box_index": box_index,
                "class_id": class_id,
                "x_center": x,
                "y_center": y,
                "width": w,
                "height": h
            })

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    f"\nMalformed boxes found: {len(bad_boxes)}"
)

print("\nDETAILS")
print("-" * 70)

for i, box in enumerate(bad_boxes, start=1):

    print(f"\n{i}. {box['image_filename']}")
    print(f"   Split      : {box['split']}")
    print(f"   Group      : {box['group']}")
    print(f"   Box index  : {box['box_index']}")
    print(f"   Class ID   : {box['class_id']}")
    print(f"   x_center   : {box['x_center']}")
    print(f"   y_center   : {box['y_center']}")
    print(f"   width      : {box['width']}")
    print(f"   height     : {box['height']}")

# ------------------------------------------------------------
# Class distribution of malformed boxes
# ------------------------------------------------------------

from collections import Counter

bad_class_counts = Counter(
    box["class_id"]
    for box in bad_boxes
)

print("\n" + "=" * 70)
print("MALFORMED BOXES BY CLASS")
print("=" * 70)

for class_id in sorted(bad_class_counts):

    print(
        f"Class {class_id:2d}: "
        f"{bad_class_counts[class_id]} box(es)"
    )

# ------------------------------------------------------------
# Split distribution
# ------------------------------------------------------------

bad_split_counts = Counter(
    box["split"]
    for box in bad_boxes
)

print("\n" + "=" * 70)
print("MALFORMED BOXES BY SPLIT")
print("=" * 70)

for split_name in ["train", "val", "test"]:

    print(
        f"{split_name:<6}: "
        f"{bad_split_counts.get(split_name, 0)}"
    )

print("\n" + "=" * 70)
print("✓ MALFORMED BOX AUDIT COMPLETE")
print("=" * 70)

AUDITING MALFORMED BOUNDING BOXES

Malformed boxes found: 12

DETAILS
----------------------------------------------------------------------

1. 0908tis3cgp3452411_120.jpeg
   Split      : train
   Group      : NDG_00111
   Box index  : 1
   Class ID   : 1
   x_center   : 0.8861979166666667
   y_center   : 0.5148148148148148
   width      : 0.22552083333333334
   height     : 0.0

2. 1108u04wol98472626_117.jpeg
   Split      : train
   Group      : NDG_03596
   Box index  : 2
   Class ID   : 1
   x_center   : 0.0
   y_center   : 0.16712962962962963
   width      : 0.0
   height     : 0.028703703703703703

3. 12087fuyrjar392539_522.jpeg
   Split      : train
   Group      : NDG_04501
   Box index  : 1
   Class ID   : 1
   x_center   : 0.7098958333333333
   y_center   : 0.014814814814814815
   width      : 0.003125
   height     : 0.0

4. 1208eskykx7w392539_279.jpeg
   Split      : train
   Group      : NDG_04791
   Box index  : 1
   Class ID   : 1
   x_center   : 0.26328125
   y_center 

In [22]:
# Drive space audit
import shutil

total, used, free = shutil.disk_usage("/content/drive")

print("=" * 60)
print("GOOGLE DRIVE STORAGE")
print("=" * 60)

print(f"Total : {total / (1024**3):.2f} GB")
print(f"Used  : {used / (1024**3):.2f} GB")
print(f"Free  : {free / (1024**3):.2f} GB")

GOOGLE DRIVE STORAGE
Total : 107.72 GB
Used  : 29.00 GB
Free  : 78.72 GB


In [23]:
# ============================================================
# Create storage-efficient YOLO annotations
from datetime import datetime

print("=" * 70)
print("CREATING DERIVED YOLO ANNOTATIONS")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

DATA_DIR = PROJECT_ROOT / "01_data"
SPLITS_DIR = DATA_DIR / "splits"
YOLO_DIR = DATA_DIR / "yolo_baseline"
LOG_DIR = PROJECT_ROOT / "logs"

YOLO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Load authoritative persisted split
# ------------------------------------------------------------

split_manifest = (
    SPLITS_DIR / "split_manifest.csv"
)

assert split_manifest.exists(), (
    f"Missing split manifest:\n{split_manifest}"
)

split_df = pd.read_csv(
    split_manifest
)

print(f"\nManifest : {split_manifest}")
print(f"Rows     : {len(split_df):,}")

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "image_id",
    "image_path",
    "annotation_path",
    "image_filename",
    "near_duplicate_group",
    "split"
]

missing = [
    c for c in required_columns
    if c not in split_df.columns
]

assert not missing, (
    f"Missing required columns: {missing}"
)

print("✓ Required columns present")

# ------------------------------------------------------------
# Expected split counts
# ------------------------------------------------------------

EXPECTED_SPLITS = {
    "train": 5326,
    "val": 668,
    "test": 662
}

actual_splits = (
    split_df["split"]
    .value_counts()
    .to_dict()
)

print("\nSPLIT COUNTS")
print("-" * 70)

for split_name in ["train", "val", "test"]:

    actual = actual_splits.get(
        split_name,
        0
    )

    expected = EXPECTED_SPLITS[
        split_name
    ]

    print(
        f"{split_name:<6}: "
        f"{actual:,} / {expected:,}"
    )

    assert actual == expected

# ------------------------------------------------------------
# Create label directories
# ------------------------------------------------------------

for split_name in [
    "train",
    "val",
    "test"
]:

    (
        YOLO_DIR
        / "labels"
        / split_name
    ).mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Conversion
# ------------------------------------------------------------

EXPECTED_CLASSES = set(range(15))

total_source_boxes = 0
total_valid_boxes = 0

excluded_boxes = []

images_processed = 0

for _, row in split_df.iterrows():

    split_name = str(
        row["split"]
    )

    image_filename = str(
        row["image_filename"]
    )

    annotation_path = Path(
        str(row["annotation_path"])
    )

    assert annotation_path.exists(), (
        f"Annotation missing:\n"
        f"{annotation_path}"
    )

    # --------------------------------------------------------
    # Output label path
    # --------------------------------------------------------

    label_path = (
        YOLO_DIR
        / "labels"
        / split_name
        / (
            Path(image_filename).stem
            + ".txt"
        )
    )

    # --------------------------------------------------------
    # Read canonical JSON
    # --------------------------------------------------------

    with open(
        annotation_path,
        "r"
    ) as f:

        annotation = json.load(f)

    assert isinstance(
        annotation,
        list
    )

    yolo_lines = []

    # --------------------------------------------------------
    # Convert every annotation
    # --------------------------------------------------------

    for box_index, box in enumerate(annotation):

        total_source_boxes += 1

        class_id = int(
            box["class_id"]
        )

        x = float(
            box["x_center"]
        )

        y = float(
            box["y_center"]
        )

        w = float(
            box["width"]
        )

        h = float(
            box["height"]
        )

        # ----------------------------------------------------
        # Exclude invalid zero-area boxes
        # ----------------------------------------------------

        if w <= 0 or h <= 0:

            excluded_boxes.append({
                "image_id":
                    row["image_id"],

                "image_filename":
                    image_filename,

                "annotation_path":
                    str(annotation_path),

                "split":
                    split_name,

                "near_duplicate_group":
                    row["near_duplicate_group"],

                "box_index":
                    box_index,

                "class_id":
                    class_id,

                "x_center":
                    x,

                "y_center":
                    y,

                "width":
                    w,

                "height":
                    h,

                "reason":
                    "zero_or_negative_area"
            })

            continue

        # ----------------------------------------------------
        # Validity assertions
        # ----------------------------------------------------

        assert class_id in EXPECTED_CLASSES

        assert 0 <= x <= 1
        assert 0 <= y <= 1
        assert 0 < w <= 1
        assert 0 < h <= 1

        # ----------------------------------------------------
        # YOLO format
        # ----------------------------------------------------

        yolo_lines.append(
            f"{class_id} "
            f"{x:.10f} "
            f"{y:.10f} "
            f"{w:.10f} "
            f"{h:.10f}"
        )

        total_valid_boxes += 1

    # --------------------------------------------------------
    # Write label
    # --------------------------------------------------------

    with open(
        label_path,
        "w"
    ) as f:

        if yolo_lines:

            f.write(
                "\n".join(yolo_lines)
                + "\n"
            )

    images_processed += 1

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONVERSION SUMMARY")
print("=" * 70)

print(
    f"Images processed       : "
    f"{images_processed:,}"
)

print(
    f"Source annotation boxes: "
    f"{total_source_boxes:,}"
)

print(
    f"Valid YOLO boxes       : "
    f"{total_valid_boxes:,}"
)

print(
    f"Excluded invalid boxes : "
    f"{len(excluded_boxes):,}"
)

print(
    f"Validation total       : "
    f"{total_valid_boxes + len(excluded_boxes):,}"
)

# ------------------------------------------------------------
# Expected values
# ------------------------------------------------------------

assert images_processed == 6656

assert total_source_boxes == 62052

assert len(excluded_boxes) == 12

assert total_valid_boxes == 62040

assert (
    total_valid_boxes
    + len(excluded_boxes)
    == total_source_boxes
)

print("\n✓ 6,656 IMAGES PROCESSED")
print("✓ 62,052 SOURCE BOXES ACCOUNTED FOR")
print("✓ 62,040 VALID BOXES CONVERTED")
print("✓ 12 INVALID BOXES EXCLUDED")

# ------------------------------------------------------------
# Save exclusion audit
# ------------------------------------------------------------

exclusion_df = pd.DataFrame(
    excluded_boxes
)

exclusion_path = (
    YOLO_DIR
    / "excluded_invalid_boxes.csv"
)

exclusion_df.to_csv(
    exclusion_path,
    index=False
)

print(
    f"\nExclusion audit:"
    f"\n{exclusion_path}"
)

# ------------------------------------------------------------
# Save conversion metadata
# ------------------------------------------------------------

conversion_metadata = {
    "created_at":
        datetime.now().isoformat(),

    "source_manifest":
        str(split_manifest),

    "source_images":
        6656,

    "source_boxes":
        62052,

    "valid_boxes":
        62040,

    "excluded_boxes":
        12,

    "classes":
        15,

    "class_ids":
        list(range(15)),

    "annotation_format":
        "JSON",

    "target_format":
        "YOLO",

    "exclusion_rule":
        "width <= 0 OR height <= 0",

    "canonical_annotations_modified":
        False,

    "storage_strategy":
        "labels_only"
}

metadata_path = (
    YOLO_DIR
    / "conversion_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        conversion_metadata,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Save experiment log
# ------------------------------------------------------------

log_path = (
    LOG_DIR
    / "baseline_yolo_annotation_conversion.txt"
)

with open(
    log_path,
    "w"
) as f:

    f.write(
        "BASELINE YOLO ANNOTATION CONVERSION\n"
    )

    f.write("=" * 70 + "\n")

    f.write(
        f"Timestamp: "
        f"{datetime.now().isoformat()}\n"
    )

    f.write(
        f"Images processed: 6656\n"
    )

    f.write(
        f"Source boxes: 62052\n"
    )

    f.write(
        f"Valid boxes: 62040\n"
    )

    f.write(
        f"Excluded boxes: 12\n"
    )

    f.write(
        "Exclusion rule: "
        "width <= 0 OR height <= 0\n"
    )

    f.write(
        "Canonical JSON annotations "
        "were NOT modified.\n"
    )

    f.write(
        "Images were NOT duplicated into "
        "Google Drive YOLO directories.\n"
    )

print(
    f"\nConversion metadata:"
    f"\n{metadata_path}"
)

print(
    f"\nExperiment log:"
    f"\n{log_path}"
)

print("\n" + "=" * 70)
print("✓ DERIVED YOLO LABELS CREATED")
print("✓ NO IMAGE DUPLICATION IN DRIVE")
print("✓ CANONICAL DATASET PRESERVED")
print("=" * 70)

CREATING DERIVED YOLO ANNOTATIONS

Manifest : /content/drive/MyDrive/dissertation_weed_detection/01_data/splits/split_manifest.csv
Rows     : 6,656
✓ Required columns present

SPLIT COUNTS
----------------------------------------------------------------------
train : 5,326 / 5,326
val   : 668 / 668
test  : 662 / 662

CONVERSION SUMMARY
Images processed       : 6,656
Source annotation boxes: 62,052
Valid YOLO boxes       : 62,040
Excluded invalid boxes : 12
Validation total       : 62,052

✓ 6,656 IMAGES PROCESSED
✓ 62,052 SOURCE BOXES ACCOUNTED FOR
✓ 62,040 VALID BOXES CONVERTED
✓ 12 INVALID BOXES EXCLUDED

Exclusion audit:
/content/drive/MyDrive/dissertation_weed_detection/01_data/yolo_baseline/excluded_invalid_boxes.csv

Conversion metadata:
/content/drive/MyDrive/dissertation_weed_detection/01_data/yolo_baseline/conversion_metadata.json

Experiment log:
/content/drive/MyDrive/dissertation_weed_detection/logs/baseline_yolo_annotation_conversion.txt

✓ DERIVED YOLO LABELS CREATED
✓ NO

In [24]:
# ============================================================
# Cell 14: YOLO Annotation Integrity Audit
from collections import Counter

print("=" * 70)
print("YOLO ANNOTATION INTEGRITY AUDIT")
print("=" * 70)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

YOLO_DIR = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
)

SPLITS_DIR = (
    PROJECT_ROOT
    / "01_data"
    / "splits"
)

split_manifest_path = (
    SPLITS_DIR
    / "split_manifest.csv"
)

split_df = pd.read_csv(
    split_manifest_path
)

# ------------------------------------------------------------
# Expected values
# ------------------------------------------------------------

EXPECTED_IMAGES = {
    "train": 5326,
    "val": 668,
    "test": 662
}

EXPECTED_BOXES = 62040
EXPECTED_CLASSES = set(range(15))

# ------------------------------------------------------------
# Audit counters
# ------------------------------------------------------------

total_label_files = 0
total_boxes = 0

invalid_rows = []
invalid_classes = []
invalid_coordinates = []

class_counter = Counter()
split_box_counter = Counter()

empty_label_files = []

# ------------------------------------------------------------
# Audit each split
# ------------------------------------------------------------

print("\nSPLIT AUDIT")
print("-" * 70)

for split_name in ["train", "val", "test"]:

    split_df_part = split_df[
        split_df["split"] == split_name
    ]

    label_dir = (
        YOLO_DIR
        / "labels"
        / split_name
    )

    assert label_dir.exists(), (
        f"Missing label directory:\n{label_dir}"
    )

    expected_images = EXPECTED_IMAGES[
        split_name
    ]

    actual_manifest_images = len(
        split_df_part
    )

    print(
        f"\n{split_name.upper()}"
    )

    print(
        f"Manifest images : "
        f"{actual_manifest_images:,}"
    )

    print(
        f"Expected images : "
        f"{expected_images:,}"
    )

    assert (
        actual_manifest_images
        == expected_images
    )

    # --------------------------------------------------------
    # Check each manifest image
    # --------------------------------------------------------

    for _, row in split_df_part.iterrows():

        image_filename = str(
            row["image_filename"]
        )

        expected_label = (
            Path(image_filename).stem
            + ".txt"
        )

        label_path = (
            label_dir
            / expected_label
        )

        if not label_path.exists():

            invalid_rows.append({
                "split": split_name,
                "image": image_filename,
                "problem": "missing_label_file"
            })

            continue

        total_label_files += 1

        with open(
            label_path,
            "r"
        ) as f:

            lines = [
                line.strip()
                for line in f
                if line.strip()
            ]

        if len(lines) == 0:

            empty_label_files.append({
                "split": split_name,
                "image": image_filename
            })

            continue

        for line_number, line in enumerate(
            lines,
            start=1
        ):

            parts = line.split()

            # ------------------------------------------------
            # YOLO row must contain 5 values
            # ------------------------------------------------

            if len(parts) != 5:

                invalid_rows.append({
                    "split": split_name,
                    "image": image_filename,
                    "line": line_number,
                    "problem":
                        "expected_5_values",
                    "content": line
                })

                continue

            try:

                class_id = int(parts[0])

                x = float(parts[1])
                y = float(parts[2])
                w = float(parts[3])
                h = float(parts[4])

            except ValueError:

                invalid_rows.append({
                    "split": split_name,
                    "image": image_filename,
                    "line": line_number,
                    "problem":
                        "non_numeric_value",
                    "content": line
                })

                continue

            total_boxes += 1
            split_box_counter[split_name] += 1

            # ------------------------------------------------
            # Class validation
            # ------------------------------------------------

            if class_id not in EXPECTED_CLASSES:

                invalid_classes.append({
                    "split": split_name,
                    "image": image_filename,
                    "line": line_number,
                    "class_id": class_id
                })

            else:

                class_counter[class_id] += 1

            # ------------------------------------------------
            # Coordinate validation
            # ------------------------------------------------

            valid_coordinates = (
                0 <= x <= 1
                and
                0 <= y <= 1
                and
                0 < w <= 1
                and
                0 < h <= 1
            )

            if not valid_coordinates:

                invalid_coordinates.append({
                    "split": split_name,
                    "image": image_filename,
                    "line": line_number,
                    "x": x,
                    "y": y,
                    "width": w,
                    "height": h
                })

    # --------------------------------------------------------
    # Report split
    # --------------------------------------------------------

    actual_labels = len(
        list(label_dir.glob("*.txt"))
    )

    print(
        f"Label files found: "
        f"{actual_labels:,}"
    )

    print(
        f"Expected labels  : "
        f"{expected_images:,}"
    )

    assert actual_labels == expected_images

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Label files checked : "
    f"{total_label_files:,}"
)

print(
    f"YOLO boxes          : "
    f"{total_boxes:,}"
)

print(
    f"Expected boxes      : "
    f"{EXPECTED_BOXES:,}"
)

print(
    f"Invalid rows        : "
    f"{len(invalid_rows):,}"
)

print(
    f"Invalid classes     : "
    f"{len(invalid_classes):,}"
)

print(
    f"Invalid coordinates : "
    f"{len(invalid_coordinates):,}"
)

print(
    f"Empty label files   : "
    f"{len(empty_label_files):,}"
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert total_label_files == 6656

assert total_boxes == EXPECTED_BOXES

assert len(invalid_rows) == 0

assert len(invalid_classes) == 0

assert len(invalid_coordinates) == 0

# Empty labels are not automatically an error,
# but we inspect them explicitly.
print("\nEMPTY LABEL FILES")

if empty_label_files:

    print(
        f"Found: "
        f"{len(empty_label_files):,}"
    )

    for item in empty_label_files[:20]:

        print(
            f"{item['split']} : "
            f"{item['image']}"
        )

else:

    print("None")

# ------------------------------------------------------------
# Split box distribution
# ------------------------------------------------------------

print("\nBOXES BY SPLIT")
print("-" * 70)

for split_name in ["train", "val", "test"]:

    print(
        f"{split_name:<6}: "
        f"{split_box_counter[split_name]:,}"
    )

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

print("\nCLASS DISTRIBUTION")
print("-" * 70)

for class_id in sorted(
    EXPECTED_CLASSES
):

    print(
        f"Class {class_id:2d}: "
        f"{class_counter[class_id]:,}"
    )

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ YOLO ANNOTATION INTEGRITY VERIFIED")
print("=" * 70)

print(
    "✓ 6,656 label files"
)

print(
    "✓ 62,040 valid YOLO boxes"
)

print(
    "✓ 15 valid classes"
)

print(
    "✓ All coordinates valid"
)

print(
    "✓ No malformed YOLO rows"
)

print(
    "✓ No missing label files"
)

YOLO ANNOTATION INTEGRITY AUDIT

SPLIT AUDIT
----------------------------------------------------------------------

TRAIN
Manifest images : 5,326
Expected images : 5,326
Label files found: 5,326
Expected labels  : 5,326

VAL
Manifest images : 668
Expected images : 668
Label files found: 668
Expected labels  : 668

TEST
Manifest images : 662
Expected images : 662
Label files found: 662
Expected labels  : 662

VALIDATION SUMMARY
Label files checked : 6,656
YOLO boxes          : 62,040
Expected boxes      : 62,040
Invalid rows        : 0
Invalid classes     : 0
Invalid coordinates : 0
Empty label files   : 8

EMPTY LABEL FILES
Found: 8
train : 0908v6m8vg8q462413_144.jpeg
train : 110805fn6u7n302458_963.jpeg
train : 11084f2my474472626_171.jpeg
train : 11088pnly5u7482628_115.jpeg
train : 1108ftmdip21472626_177.jpeg
train : 1108gdueg967272453_873.jpeg
train : 1108rex9sz4s422408_147.jpeg
train : 160816jm63c2062404_112.jpeg

BOXES BY SPLIT
------------------------------------------------------

In [25]:
# ============================================================
# Cell 15: Visual Verification of YOLO Bounding Boxes
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import random
import os

print("=" * 70)
print("VISUAL YOLO LABEL VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

YOLO_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
)

SPLIT_MANIFEST = (
    PROJECT_ROOT
    / "01_data"
    / "splits"
    / "split_manifest.csv"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)

# Find the most recent baseline experiment directory
experiment_dirs = sorted(
    RESULTS_ROOT.glob("baseline_yolo_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if experiment_dirs:
    EXPERIMENT_RESULTS = experiment_dirs[0]
else:
    EXPERIMENT_RESULTS = RESULTS_ROOT / "baseline_yolo_visual_audit"

EXPERIMENT_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)

print(f"\nYOLO root       : {YOLO_ROOT}")
print(f"Split manifest  : {SPLIT_MANIFEST}")
print(f"Results         : {EXPERIMENT_RESULTS}")

# ------------------------------------------------------------
# 2. LOAD MANIFEST
# ------------------------------------------------------------

df = pd.read_csv(
    SPLIT_MANIFEST
)

required_columns = [
    "image_id",
    "image_path",
    "image_filename",
    "split"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("\nMANIFEST")
print("-" * 70)

for split_name in ["train", "val", "test"]:

    count = (
        df["split"] == split_name
    ).sum()

    print(
        f"{split_name:<6}: {count:,} images"
    )

# ------------------------------------------------------------
# 3. YOLO LABEL READER
# ------------------------------------------------------------

def read_yolo_labels(label_path):

    boxes = []

    if not label_path.exists():
        return boxes

    with open(
        label_path,
        "r"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])

            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            boxes.append(
                (
                    class_id,
                    x_center,
                    y_center,
                    width,
                    height
                )
            )

    return boxes


# ------------------------------------------------------------
# 4. DRAW YOLO BOXES
# ------------------------------------------------------------

def create_annotated_image(
    image_path,
    label_path
):

    image = Image.open(
        image_path
    ).convert("RGB")

    draw = ImageDraw.Draw(
        image
    )

    image_width, image_height = (
        image.size
    )

    boxes = read_yolo_labels(
        label_path
    )

    for (
        class_id,
        xc,
        yc,
        w,
        h
    ) in boxes:

        # Convert normalized YOLO coordinates
        # to pixel coordinates

        x1 = int(
            (xc - w / 2)
            * image_width
        )

        y1 = int(
            (yc - h / 2)
            * image_height
        )

        x2 = int(
            (xc + w / 2)
            * image_width
        )

        y2 = int(
            (yc + h / 2)
            * image_height
        )

        # Clamp to image boundaries
        x1 = max(
            0,
            min(x1, image_width - 1)
        )

        y1 = max(
            0,
            min(y1, image_height - 1)
        )

        x2 = max(
            0,
            min(x2, image_width - 1)
        )

        y2 = max(
            0,
            min(y2, image_height - 1)
        )

        # Draw bounding box
        draw.rectangle(
            [
                x1,
                y1,
                x2,
                y2
            ],
            outline="red",
            width=max(
                2,
                image_width // 400
            )
        )

        # Class label
        label = f"class {class_id}"

        # Approximate label size
        text_height = max(
            14,
            image_height // 60
        )

        # Draw label background
        text_x = x1
        text_y = max(
            0,
            y1 - text_height - 4
        )

        draw.rectangle(
            [
                text_x,
                text_y,
                text_x + 80,
                y1
            ],
            fill="red"
        )

        draw.text(
            (
                text_x + 3,
                text_y + 2
            ),
            label,
            fill="white"
        )

    return image, len(boxes)


# ------------------------------------------------------------
# 5. CREATE CONTACT SHEETS
# ------------------------------------------------------------

def create_contact_sheet(
    split_name,
    sample_size=12,
    seed=42
):

    split_df = df[
        df["split"] == split_name
    ].copy()

    rng = random.Random(seed)

    sample_size = min(
        sample_size,
        len(split_df)
    )

    sampled_rows = rng.sample(
        list(
            split_df.iterrows()
        ),
        sample_size
    )

    # 4 columns x 3 rows
    cols = 4
    rows = int(
        np.ceil(
            sample_size / cols
        )
    )

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 5 * rows)
    )

    axes = np.array(
        axes
    ).reshape(-1)

    total_boxes = 0
    images_with_boxes = 0
    images_without_boxes = 0

    for ax, (_, row) in zip(
        axes,
        sampled_rows
    ):

        image_path = Path(
            row["image_path"]
        )

        image_filename = str(
            row["image_filename"]
        )

        label_path = (
            YOLO_ROOT
            / "labels"
            / split_name
            / (
                Path(
                    image_filename
                ).stem
                + ".txt"
            )
        )

        if not image_path.exists():

            ax.text(
                0.5,
                0.5,
                "IMAGE NOT FOUND",
                ha="center",
                va="center"
            )

            ax.axis("off")

            continue

        image, box_count = (
            create_annotated_image(
                image_path,
                label_path
            )
        )

        total_boxes += box_count

        if box_count > 0:
            images_with_boxes += 1
        else:
            images_without_boxes += 1

        ax.imshow(
            image
        )

        ax.set_title(
            f"{image_filename}\n"
            f"{box_count} boxes",
            fontsize=9
        )

        ax.axis("off")

    # Hide unused axes
    for ax in axes[
        sample_size:
    ]:

        ax.axis("off")

    fig.suptitle(
        f"YOLO Bounding Box Verification — "
        f"{split_name.upper()}",
        fontsize=18,
        fontweight="bold"
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.96
        ]
    )

    output_path = (
        EXPERIMENT_RESULTS
        / f"yolo_visual_audit_{split_name}.png"
    )

    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(
        fig
    )

    return {
        "split": split_name,
        "sampled_images": sample_size,
        "images_with_boxes": images_with_boxes,
        "images_without_boxes": images_without_boxes,
        "boxes_in_sample": total_boxes,
        "output": str(output_path)
    }


# ------------------------------------------------------------
# 6. GENERATE ALL THREE SHEETS
# ------------------------------------------------------------

print("\nGENERATING CONTACT SHEETS")
print("-" * 70)

audit_results = []

for split_name in [
    "train",
    "val",
    "test"
]:

    result = create_contact_sheet(
        split_name=split_name,
        sample_size=12,
        seed=42
    )

    audit_results.append(
        result
    )

    print(
        f"\n{split_name.upper()}"
    )

    print(
        f"Sampled images : "
        f"{result['sampled_images']}"
    )

    print(
        f"Images w/ boxes: "
        f"{result['images_with_boxes']}"
    )

    print(
        f"Images no boxes : "
        f"{result['images_without_boxes']}"
    )

    print(
        f"Boxes in sample : "
        f"{result['boxes_in_sample']}"
    )

    print(
        f"Saved           : "
        f"{result['output']}"
    )

# ------------------------------------------------------------
# 7. SAVE AUDIT METADATA
# ------------------------------------------------------------

audit_metadata = {
    "experiment": "baseline_yolo",
    "seed": 42,
    "sample_size_per_split": 12,
    "splits": audit_results
}

metadata_path = (
    EXPERIMENT_RESULTS
    / "yolo_visual_audit_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        audit_metadata,
        f,
        indent=2
    )

print("\n" + "=" * 70)
print("✓ VISUAL YOLO AUDIT COMPLETE")
print("=" * 70)

print(
    f"Metadata saved:\n{metadata_path}"
)

print(
    "\nReview the three generated contact sheets "
    "before starting training."
)

VISUAL YOLO LABEL VERIFICATION

YOLO root       : /content/drive/MyDrive/dissertation_weed_detection/01_data/yolo_baseline
Split manifest  : /content/drive/MyDrive/dissertation_weed_detection/01_data/splits/split_manifest.csv
Results         : /content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_065727

MANIFEST
----------------------------------------------------------------------
train : 5,326 images
val   : 668 images
test  : 662 images

GENERATING CONTACT SHEETS
----------------------------------------------------------------------

TRAIN
Sampled images : 12
Images w/ boxes: 12
Images no boxes : 0
Boxes in sample : 142
Saved           : /content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_065727/yolo_visual_audit_train.png

VAL
Sampled images : 12
Images w/ boxes: 12
Images no boxes : 0
Boxes in sample : 122
Saved           : /content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_065727/yolo_visual

Baseline YOLO

In [26]:
# ============================================================
# Baseline YOLO Experiment Configuration

import platform
import subprocess

print("=" * 70)
print("BASELINE YOLO EXPERIMENT CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

YOLO_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
)

SPLIT_MANIFEST = (
    PROJECT_ROOT
    / "01_data"
    / "splits"
    / "split_manifest.csv"
)

CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
)

LOG_DIR = (
    PROJECT_ROOT
    / "logs"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

for directory in [
    CONFIG_DIR,
    LOG_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# 2. EXPERIMENT ID
# ------------------------------------------------------------

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

EXPERIMENT_ID = (
    f"baseline_yolo_{timestamp}"
)

EXPERIMENT_RESULTS = (
    RESULTS_DIR
    / EXPERIMENT_ID
)

EXPERIMENT_CHECKPOINTS = (
    CHECKPOINT_DIR
    / EXPERIMENT_ID
)

EXPERIMENT_LOGS = (
    LOG_DIR
    / EXPERIMENT_ID
)

EXPERIMENT_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)

EXPERIMENT_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True
)

EXPERIMENT_LOGS.mkdir(
    parents=True,
    exist_ok=True
)

print("\nEXPERIMENT")
print("-" * 70)
print(f"ID        : {EXPERIMENT_ID}")
print(f"Results   : {EXPERIMENT_RESULTS}")
print(f"Checkpoints: {EXPERIMENT_CHECKPOINTS}")
print(f"Logs      : {EXPERIMENT_LOGS}")

# ------------------------------------------------------------
# 3. DATASET CONFIGURATION
# ------------------------------------------------------------

DATASET_CONFIG = {

    "dataset_name":
        "MH_Weed16_weed_detection",

    "source":
        "Project-AgML/MH_Weed16_weed_detection",

    "dataset_root":
        str(YOLO_ROOT),

    "split_manifest":
        str(SPLIT_MANIFEST),

    "num_classes":
        15,

    "classes":
        [str(i) for i in range(15)],

    "train_images":
        5326,

    "val_images":
        668,

    "test_images":
        662,

    "total_images":
        6656,

    "source_boxes":
        62052,

    "valid_yolo_boxes":
        62040,

    "excluded_invalid_boxes":
        12,

    "near_duplicate_groups":
        6439,

    "group_leakage":
        False
}

# ------------------------------------------------------------
# 4. BASELINE MODEL CONFIGURATION
# ------------------------------------------------------------
#
# These are intentionally kept in one place.
# We will NOT start training in this cell.
#
# MODEL_NAME is left explicit so that the training cell uses
# exactly the model recorded here.

MODEL_CONFIG = {

    "framework":
        "Ultralytics YOLO",

    "model":
        "yolo11n.pt",

    "pretrained":
        True,

    "num_classes":
        15,

    "task":
        "detect"
}

# ------------------------------------------------------------
# 5. TRAINING CONFIGURATION
# ------------------------------------------------------------
#
# Conservative baseline settings suitable for Colab.
# They can be changed before training, but once training
# begins this configuration should be treated as immutable.

TRAINING_CONFIG = {

    "epochs":
        100,

    "image_size":
        640,

    "batch":
        16,

    "device":
        0,

    "workers":
        2,

    "seed":
        42,

    "deterministic":
        True,

    "patience":
        20,

    "save":
        True,

    "plots":
        True,

    "cache":
        False
}

# ------------------------------------------------------------
# 6. AUGMENTATION CONFIGURATION
# ------------------------------------------------------------
#
# For the first baseline, keep augmentation controlled.
# We do not want the baseline itself to contain aggressive
# experimental transformations.

AUGMENTATION_CONFIG = {

    "hsv_h":
        0.015,

    "hsv_s":
        0.7,

    "hsv_v":
        0.4,

    "degrees":
        0.0,

    "translate":
        0.1,

    "scale":
        0.5,

    "shear":
        0.0,

    "perspective":
        0.0,

    "flipud":
        0.0,

    "fliplr":
        0.5,

    "mosaic":
        1.0,

    "mixup":
        0.0,

    "copy_paste":
        0.0
}

# ------------------------------------------------------------
# 7. RANDOM SEEDS
# ------------------------------------------------------------

random.seed(42)
np.random.seed(42)

# ------------------------------------------------------------
# 8. ENVIRONMENT INFORMATION
# ------------------------------------------------------------

def get_package_version(package_name):

    try:

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "show",
                package_name
            ],
            capture_output=True,
            text=True
        )

        for line in result.stdout.splitlines():

            if line.startswith(
                "Version:"
            ):

                return line.split(
                    ":",
                    1
                )[1].strip()

    except Exception:

        pass

    return None


environment = {

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "system":
        platform.system(),

    "machine":
        platform.machine(),

    "ultralytics_version":
        get_package_version(
            "ultralytics"
        ),

    "torch_version":
        get_package_version(
            "torch"
        ),

    "numpy_version":
        get_package_version(
            "numpy"
        ),

    "pandas_version":
        get_package_version(
            "pandas"
        ),

    "opencv_version":
        get_package_version(
            "opencv-python"
        )
}

# ------------------------------------------------------------
# 9. COMPLETE CONFIGURATION
# ------------------------------------------------------------

experiment_config = {

    "experiment_id":
        EXPERIMENT_ID,

    "created":
        datetime.now().isoformat(),

    "dataset":
        DATASET_CONFIG,

    "model":
        MODEL_CONFIG,

    "training":
        TRAINING_CONFIG,

    "augmentation":
        AUGMENTATION_CONFIG,

    "environment":
        environment,

    "paths": {

        "project_root":
            str(PROJECT_ROOT),

        "yolo_root":
            str(YOLO_ROOT),

        "split_manifest":
            str(SPLIT_MANIFEST),

        "results":
            str(EXPERIMENT_RESULTS),

        "checkpoints":
            str(EXPERIMENT_CHECKPOINTS),

        "logs":
            str(EXPERIMENT_LOGS)
    }
}

# ------------------------------------------------------------
# 10. SAVE CONFIGURATION
# ------------------------------------------------------------

config_path = (
    EXPERIMENT_RESULTS
    / "baseline_experiment_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        experiment_config,
        f,
        indent=2
    )

# Also save a copy in the central configs directory
central_config_path = (
    CONFIG_DIR
    / f"{EXPERIMENT_ID}.json"
)

with open(
    central_config_path,
    "w"
) as f:

    json.dump(
        experiment_config,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 11. PRINT CONFIGURATION
# ------------------------------------------------------------

print("\nDATASET")
print("-" * 70)

print(
    f"Images       : "
    f"{DATASET_CONFIG['total_images']:,}"
)

print(
    f"Classes      : "
    f"{DATASET_CONFIG['num_classes']}"
)

print(
    f"Valid boxes  : "
    f"{DATASET_CONFIG['valid_yolo_boxes']:,}"
)

print(
    f"Group leakage: "
    f"{DATASET_CONFIG['group_leakage']}"
)

print("\nMODEL")
print("-" * 70)

for key, value in MODEL_CONFIG.items():

    print(
        f"{key:<18}: {value}"
    )

print("\nTRAINING")
print("-" * 70)

for key, value in TRAINING_CONFIG.items():

    print(
        f"{key:<18}: {value}"
    )

print("\nENVIRONMENT")
print("-" * 70)

print(
    f"Python     : "
    f"{environment['python_version'].split()[0]}"
)

print(
    f"Ultralytics: "
    f"{environment['ultralytics_version']}"
)

print(
    f"PyTorch    : "
    f"{environment['torch_version']}"
)

print(
    f"NumPy      : "
    f"{environment['numpy_version']}"
)

print("\n" + "=" * 70)
print("✓ BASELINE CONFIGURATION SAVED")
print("=" * 70)

print(
    f"\nExperiment config:\n{config_path}"
)

print(
    f"\nCentral config copy:\n{central_config_path}"
)

BASELINE YOLO EXPERIMENT CONFIGURATION

EXPERIMENT
----------------------------------------------------------------------
ID        : baseline_yolo_20260909_092338
Results   : /content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_092338
Checkpoints: /content/drive/MyDrive/dissertation_weed_detection/checkpoints/baseline_yolo_20260909_092338
Logs      : /content/drive/MyDrive/dissertation_weed_detection/logs/baseline_yolo_20260909_092338

DATASET
----------------------------------------------------------------------
Images       : 6,656
Classes      : 15
Valid boxes  : 62,040
Group leakage: False

MODEL
----------------------------------------------------------------------
framework         : Ultralytics YOLO
model             : yolo11n.pt
pretrained        : True
num_classes       : 15
task              : detect

TRAINING
----------------------------------------------------------------------
epochs            : 100
image_size        : 640
batch             :

In [1]:
# ============================================================
# Colab GPU RUNTIME CHECK
# ============================================================

import os
import sys
import subprocess

print("=" * 70)
print("COLAB GPU RUNTIME CHECK")
print("=" * 70)

print("\nCOLAB ENVIRONMENT")
print("-" * 70)
print("COLAB_GPU             :", os.environ.get("COLAB_GPU"))
print("CUDA_VISIBLE_DEVICES  :", os.environ.get("CUDA_VISIBLE_DEVICES"))

print("\nNVIDIA-SMI")
print("-" * 70)

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )

    print("Return code:", result.returncode)

    if result.returncode == 0:
        print(result.stdout)
    else:
        print(result.stderr)

except FileNotFoundError:
    print("nvidia-smi is NOT available in this runtime.")

print("\nPYTORCH")
print("-" * 70)

try:
    import torch

    print("PyTorch version  :", torch.__version__)
    print("CUDA available   :", torch.cuda.is_available())
    print("CUDA version     :", torch.version.cuda)

    if torch.cuda.is_available():
        print(
            "GPU              :",
            torch.cuda.get_device_name(0)
        )
    else:
        print("GPU              : NONE")

except Exception as e:
    print("PyTorch check failed:", repr(e))

print("\n" + "=" * 70)

COLAB GPU RUNTIME CHECK

COLAB ENVIRONMENT
----------------------------------------------------------------------
COLAB_GPU             : 1
CUDA_VISIBLE_DEVICES  : None

NVIDIA-SMI
----------------------------------------------------------------------
Return code: 0
Wed Sep  9 09:53:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00

In [2]:
# ============================================================
# Cell 18: Install and Verify Ultralytics

print("=" * 70)
print("INSTALLING ULTRALYTICS YOLO ENVIRONMENT")
print("=" * 70)

# ------------------------------------------------------------
# Install Ultralytics
# ------------------------------------------------------------

print("\nInstalling Ultralytics...")
print("-" * 70)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics"
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:

    print("✗ Installation failed")
    print(result.stderr)

    raise RuntimeError(
        "Ultralytics installation failed."
    )

print("✓ Ultralytics installation completed")

# ------------------------------------------------------------
# Import and verify
# ------------------------------------------------------------

print("\nVERIFYING INSTALLATION")
print("-" * 70)

import torch
import ultralytics

print(
    "Ultralytics version :",
    ultralytics.__version__
)

print(
    "PyTorch version     :",
    torch.__version__
)

print(
    "CUDA available      :",
    torch.cuda.is_available()
)

print(
    "CUDA version        :",
    torch.version.cuda
)

if torch.cuda.is_available():

    print(
        "GPU                 :",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory          :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )

else:

    raise RuntimeError(
        "CUDA is not available. "
        "Do NOT continue to training."
    )

# ------------------------------------------------------------
# YOLO import
# ------------------------------------------------------------

try:

    from ultralytics import YOLO

    print(
        "\n✓ Ultralytics YOLO API imported successfully"
    )

except Exception as e:

    print(
        "\n✗ Failed to import YOLO:",
        repr(e)
    )

    raise

print("\n" + "=" * 70)
print("✓ YOLO ENVIRONMENT READY")
print("=" * 70)

INSTALLING ULTRALYTICS YOLO ENVIRONMENT

Installing Ultralytics...
----------------------------------------------------------------------
✓ Ultralytics installation completed

VERIFYING INSTALLATION
----------------------------------------------------------------------
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics version : 8.4.144
PyTorch version     : 2.11.0+cu128
CUDA available      : True
CUDA version        : 12.8
GPU                 : Tesla T4
GPU memory          : 14.56 GB

✓ Ultralytics YOLO API imported successfully

✓ YOLO ENVIRONMENT READY


In [5]:
# Mount Google Drive
from google.colab import drive

print("=" * 70)
print("MOUNTING GOOGLE DRIVE")
print("=" * 70)

drive.mount("/content/drive")

print("\n✓ Google Drive mounted")

MOUNTING GOOGLE DRIVE
Mounted at /content/drive

✓ Google Drive mounted


In [6]:
# ============================================================
# Cell 19: Re-establish Project Paths + Save Environment
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import platform
import subprocess
import sys

import torch
import ultralytics
import numpy
import pandas
import PIL
import cv2

print("=" * 70)
print("RE-ESTABLISHING PROJECT + SAVING YOLO ENVIRONMENT")
print("=" * 70)

# ------------------------------------------------------------
# 1. PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found:\n{PROJECT_ROOT}"
    )

print("\nProject root:")
print(PROJECT_ROOT)

# ------------------------------------------------------------
# 2. IMPORTANT PROJECT DIRECTORIES
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "01_data"
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"
YOLO_ROOT = DATA_DIR / "yolo_baseline"

ENV_DIR = PROJECT_ROOT / "environment"
CONFIG_DIR = PROJECT_ROOT / "configs"
LOG_DIR = PROJECT_ROOT / "logs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"

for path in [
    DATA_DIR,
    PROCESSED_DIR,
    SPLITS_DIR,
    YOLO_ROOT,
    ENV_DIR,
    CONFIG_DIR,
    LOG_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required project directory missing:\n{path}"
        )

print("\nPROJECT DIRECTORIES")
print("-" * 70)

for name, path in [
    ("01_data", DATA_DIR),
    ("processed", PROCESSED_DIR),
    ("splits", SPLITS_DIR),
    ("yolo_baseline", YOLO_ROOT),
    ("environment", ENV_DIR),
    ("configs", CONFIG_DIR),
    ("logs", LOG_DIR),
    ("checkpoints", CHECKPOINT_DIR),
    ("results", RESULTS_DIR)
]:
    print(f"{name:<18}: ✓")

# ------------------------------------------------------------
# 3. SPLIT MANIFEST
# ------------------------------------------------------------

SPLIT_MANIFEST = (
    SPLITS_DIR / "split_manifest.csv"
)

if not SPLIT_MANIFEST.exists():
    raise FileNotFoundError(
        f"Split manifest not found:\n{SPLIT_MANIFEST}"
    )

print("\nSplit manifest:")
print(SPLIT_MANIFEST)

# ------------------------------------------------------------
# 4. CREATE EXPERIMENT ID
# ------------------------------------------------------------

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

EXPERIMENT_ID = (
    f"baseline_yolo_{timestamp}"
)

EXPERIMENT_RESULTS = (
    RESULTS_DIR / EXPERIMENT_ID
)

EXPERIMENT_CHECKPOINTS = (
    CHECKPOINT_DIR / EXPERIMENT_ID
)

EXPERIMENT_LOGS = (
    LOG_DIR / EXPERIMENT_ID
)

for path in [
    EXPERIMENT_RESULTS,
    EXPERIMENT_CHECKPOINTS,
    EXPERIMENT_LOGS
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )

print("\nEXPERIMENT")
print("-" * 70)
print(f"ID         : {EXPERIMENT_ID}")
print(f"Results    : {EXPERIMENT_RESULTS}")
print(f"Checkpoints: {EXPERIMENT_CHECKPOINTS}")
print(f"Logs       : {EXPERIMENT_LOGS}")

# ------------------------------------------------------------
# 5. GPU INFORMATION
# ------------------------------------------------------------

gpu_available = torch.cuda.is_available()

if not gpu_available:
    raise RuntimeError(
        "CUDA is not available. "
        "Do not continue with YOLO training."
    )

gpu_name = torch.cuda.get_device_name(0)

gpu_memory_gb = round(
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3),
    2
)

# ------------------------------------------------------------
# 6. NVIDIA DRIVER INFORMATION
# ------------------------------------------------------------

nvidia_smi_output = None

try:

    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        nvidia_smi_output = result.stdout

except Exception as e:

    nvidia_smi_output = str(e)

# ------------------------------------------------------------
# 7. ENVIRONMENT RECORD
# ------------------------------------------------------------

environment_record = {

    "recorded_at":
        datetime.now().isoformat(),

    "python": {
        "version":
            platform.python_version(),

        "executable":
            sys.executable
    },

    "hardware": {
        "gpu":
            gpu_name,

        "gpu_memory_gb":
            gpu_memory_gb,

        "gpu_count":
            torch.cuda.device_count()
    },

    "cuda": {
        "pytorch_cuda":
            torch.version.cuda,

        "cuda_available":
            gpu_available
    },

    "software": {

        "ultralytics":
            ultralytics.__version__,

        "torch":
            torch.__version__,

        "numpy":
            numpy.__version__,

        "pandas":
            pandas.__version__,

        "pillow":
            PIL.__version__,

        "opencv":
            cv2.__version__
    },

    "nvidia_smi":
        nvidia_smi_output
}

# ------------------------------------------------------------
# 8. SAVE ENVIRONMENT JSON
# ------------------------------------------------------------

environment_json = (
    ENV_DIR / "baseline_yolo_environment.json"
)

with open(
    environment_json,
    "w"
) as f:

    json.dump(
        environment_record,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 9. SAVE PIP FREEZE
# ------------------------------------------------------------

freeze_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze"
    ],
    capture_output=True,
    text=True
)

freeze_path = (
    ENV_DIR / "baseline_yolo_pip_freeze.txt"
)

with open(
    freeze_path,
    "w"
) as f:

    f.write(
        freeze_result.stdout
    )

# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

print("\nVERIFIED ENVIRONMENT")
print("-" * 70)

print(
    f"Python       : {platform.python_version()}"
)

print(
    f"Ultralytics  : {ultralytics.__version__}"
)

print(
    f"PyTorch      : {torch.__version__}"
)

print(
    f"CUDA         : {torch.version.cuda}"
)

print(
    f"GPU          : {gpu_name}"
)

print(
    f"GPU memory   : {gpu_memory_gb} GB"
)

print("\nSAVED FILES")
print("-" * 70)

print(
    f"Environment JSON:\n{environment_json}"
)

print(
    f"\nPip freeze:\n{freeze_path}"
)

print("\n" + "=" * 70)
print("✓ ENVIRONMENT SNAPSHOT SAVED")
print("=" * 70)

RE-ESTABLISHING PROJECT + SAVING YOLO ENVIRONMENT

Project root:
/content/drive/MyDrive/dissertation_weed_detection

PROJECT DIRECTORIES
----------------------------------------------------------------------
01_data           : ✓
processed         : ✓
splits            : ✓
yolo_baseline     : ✓
environment       : ✓
configs           : ✓
logs              : ✓
checkpoints       : ✓
results           : ✓

Split manifest:
/content/drive/MyDrive/dissertation_weed_detection/01_data/splits/split_manifest.csv

EXPERIMENT
----------------------------------------------------------------------
ID         : baseline_yolo_20260909_095854
Results    : /content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_095854
Checkpoints: /content/drive/MyDrive/dissertation_weed_detection/checkpoints/baseline_yolo_20260909_095854
Logs       : /content/drive/MyDrive/dissertation_weed_detection/logs/baseline_yolo_20260909_095854

VERIFIED ENVIRONMENT
-------------------------------------

In [7]:
# Cell 20: Create YOLO Dataset Configuration
import yaml

print("=" * 70)
print("CREATING YOLO DATASET CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

YOLO_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
)

DATA_YAML = YOLO_ROOT / "data.yaml"

# ------------------------------------------------------------
# Verify required directories
# ------------------------------------------------------------

required_dirs = [
    YOLO_ROOT / "train_images",
    YOLO_ROOT / "val_images",
    YOLO_ROOT / "test_images",
    YOLO_ROOT / "train_labels",
    YOLO_ROOT / "val_labels",
    YOLO_ROOT / "test_labels",
]

print("\nYOLO DATASET ROOT")
print("-" * 70)
print(YOLO_ROOT)

for path in required_dirs:

    if not path.exists():
        raise FileNotFoundError(
            f"Required YOLO directory missing:\n{path}"
        )

print("\n✓ All YOLO directories verified")

# ------------------------------------------------------------
# Class names
# ------------------------------------------------------------
#
# IMPORTANT:
# Keep these IDs aligned with the existing annotations.
# Do not reorder them.
# ------------------------------------------------------------

class_names = {
    0: "class_0",
    1: "class_1",
    2: "class_2",
    3: "class_3",
    4: "class_4",
    5: "class_5",
    6: "class_6",
    7: "class_7",
    8: "class_8",
    9: "class_9",
    10: "class_10",
    11: "class_11",
    12: "class_12",
    13: "class_13",
    14: "class_14",
}

# ------------------------------------------------------------
# YOLO configuration
# ------------------------------------------------------------

data_config = {

    "path": str(YOLO_ROOT),

    "train": "train_images",

    "val": "val_images",

    "test": "test_images",

    "nc": 15,

    "names": class_names
}

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

with open(
    DATA_YAML,
    "w"
) as f:

    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

with open(
    DATA_YAML,
    "r"
) as f:

    loaded_config = yaml.safe_load(f)

assert loaded_config["nc"] == 15
assert len(loaded_config["names"]) == 15

print("\nDATASET CONFIGURATION")
print("-" * 70)

print(
    f"Classes : {loaded_config['nc']}"
)

print(
    f"Train   : {loaded_config['train']}"
)

print(
    f"Val     : {loaded_config['val']}"
)

print(
    f"Test    : {loaded_config['test']}"
)

print("\nCLASS IDs")
print("-" * 70)

for class_id, name in class_names.items():
    print(f"{class_id:>2} : {name}")

print("\nSaved:")
print(DATA_YAML)

print("\n" + "=" * 70)
print("✓ YOLO DATASET CONFIGURATION CREATED")
print("=" * 70)

CREATING YOLO DATASET CONFIGURATION

YOLO DATASET ROOT
----------------------------------------------------------------------
/content/drive/MyDrive/dissertation_weed_detection/01_data/yolo_baseline


FileNotFoundError: Required YOLO directory missing:
/content/drive/MyDrive/dissertation_weed_detection/01_data/yolo_baseline/train_images

In [8]:
# Cell 20A: Inspect Existing YOLO Dataset Structure

YOLO_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
)

print("=" * 70)
print("EXISTING YOLO DATASET STRUCTURE")
print("=" * 70)

print("\nYOLO ROOT:")
print(YOLO_ROOT)

if not YOLO_ROOT.exists():
    raise FileNotFoundError(
        f"YOLO root does not exist:\n{YOLO_ROOT}"
    )

print("\nDIRECT CONTENTS")
print("-" * 70)

for item in sorted(
    YOLO_ROOT.iterdir(),
    key=lambda x: (not x.is_dir(), x.name.lower())
):
    if item.is_dir():
        print(f"DIR  : {item.name}")
    else:
        print(f"FILE : {item.name}")

print("\nRELEVANT SUBDIRECTORIES")
print("-" * 70)

for name in [
    "train_images",
    "val_images",
    "test_images",
    "train_labels",
    "val_labels",
    "test_labels",
]:
    path = YOLO_ROOT / name
    print(
        f"{name:<15}: "
        f"{'✓ EXISTS' if path.exists() else '✗ NOT FOUND'}"
    )

print("\n" + "=" * 70)
print("✓ STRUCTURE INSPECTION COMPLETE")
print("=" * 70)

EXISTING YOLO DATASET STRUCTURE

YOLO ROOT:
/content/drive/MyDrive/dissertation_weed_detection/01_data/yolo_baseline

DIRECT CONTENTS
----------------------------------------------------------------------
DIR  : labels
FILE : conversion_metadata.json
FILE : excluded_invalid_boxes.csv

RELEVANT SUBDIRECTORIES
----------------------------------------------------------------------
train_images   : ✗ NOT FOUND
val_images     : ✗ NOT FOUND
test_images    : ✗ NOT FOUND
train_labels   : ✗ NOT FOUND
val_labels     : ✗ NOT FOUND
test_labels    : ✗ NOT FOUND

✓ STRUCTURE INSPECTION COMPLETE


In [10]:
# ============================================================
# Cell 20: Verify Image / YOLO Label Correspondence
from pathlib import Path
import pandas as pd

print("=" * 70)
print("VERIFYING IMAGE ↔ YOLO LABEL CORRESPONDENCE")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SPLIT_MANIFEST = (
    PROJECT_ROOT
    / "01_data"
    / "splits"
    / "split_manifest.csv"
)

YOLO_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
)

LABEL_ROOT = YOLO_ROOT / "labels"

# ------------------------------------------------------------
# Load manifest
# ------------------------------------------------------------

df = pd.read_csv(SPLIT_MANIFEST)

print("\nManifest rows :", len(df))

required_columns = [
    "image_id",
    "image_path",
    "annotation_path",
    "image_filename",
    "near_duplicate_group",
    "split"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing manifest columns: {missing}"
    )

# ------------------------------------------------------------
# Verify splits
# ------------------------------------------------------------

print("\nMANIFEST SPLITS")
print("-" * 70)

for split in ["train", "val", "test"]:

    subset = df[df["split"] == split]

    print(
        f"{split:<6}: "
        f"{len(subset):,} images"
    )

# ------------------------------------------------------------
# Verify label directories
# ------------------------------------------------------------

print("\nLABEL DIRECTORIES")
print("-" * 70)

for split in ["train", "val", "test"]:

    path = LABEL_ROOT / split

    if not path.exists():
        raise FileNotFoundError(
            f"Missing label directory:\n{path}"
        )

    count = len(
        list(path.glob("*.txt"))
    )

    print(
        f"{split:<6}: "
        f"{count:,} label files"
    )

# ------------------------------------------------------------
# Match labels to manifest
# ------------------------------------------------------------

print("\nLABEL ↔ IMAGE MATCHING")
print("-" * 70)

for split in ["train", "val", "test"]:

    subset = df[
        df["split"] == split
    ]

    missing_labels = []

    for filename in subset[
        "image_filename"
    ]:

        stem = Path(filename).stem

        label_path = (
            LABEL_ROOT
            / split
            / f"{stem}.txt"
        )

        if not label_path.exists():
            missing_labels.append(
                filename
            )

    print(
        f"{split:<6}: "
        f"missing labels = "
        f"{len(missing_labels)}"
    )

    if missing_labels:
        print(
            "First missing:",
            missing_labels[:5]
        )

# ------------------------------------------------------------
# Verify canonical image paths
# ------------------------------------------------------------

print("\nCANONICAL IMAGE PATH CHECK")
print("-" * 70)

missing_images = []

for image_path in df["image_path"]:

    if not Path(image_path).exists():
        missing_images.append(image_path)

print(
    "Images found   :",
    f"{len(df) - len(missing_images):,}",
    "/",
    f"{len(df):,}"
)

print(
    "Images missing :",
    f"{len(missing_images):,}"
)

if missing_images:
    print(
        "\nFirst missing image:"
    )
    print(missing_images[0])

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

assert len(missing_images) == 0

print("\n" + "=" * 70)
print("✓ IMAGE / LABEL CORRESPONDENCE VERIFIED")
print("=" * 70)

VERIFYING IMAGE ↔ YOLO LABEL CORRESPONDENCE

Manifest rows : 6656

MANIFEST SPLITS
----------------------------------------------------------------------
train : 5,326 images
val   : 668 images
test  : 662 images

LABEL DIRECTORIES
----------------------------------------------------------------------
train : 5,326 label files
val   : 668 label files
test  : 662 label files

LABEL ↔ IMAGE MATCHING
----------------------------------------------------------------------
train : missing labels = 0
val   : missing labels = 0
test  : missing labels = 0

CANONICAL IMAGE PATH CHECK
----------------------------------------------------------------------
Images found   : 6,656 / 6,656
Images missing : 0

✓ IMAGE / LABEL CORRESPONDENCE VERIFIED


In [11]:
# ============================================================
# Cell 21: Zero-Copy YOLO Dataset Preflight
# ============================================================

from pathlib import Path
import pandas as pd
import random

print("=" * 70)
print("ZERO-COPY YOLO DATASET PREFLIGHT")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SPLIT_MANIFEST = (
    PROJECT_ROOT
    / "01_data"
    / "splits"
    / "split_manifest.csv"
)

YOLO_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
)

LABEL_ROOT = (
    YOLO_ROOT
    / "labels"
)

# ------------------------------------------------------------
# Load manifest
# ------------------------------------------------------------

df = pd.read_csv(SPLIT_MANIFEST)

print("\nDATASET")
print("-" * 70)
print(f"Total images : {len(df):,}")
print(f"Classes      : 15")

# ------------------------------------------------------------
# Check each split
# ------------------------------------------------------------

random.seed(42)

split_samples = {}

for split in ["train", "val", "test"]:

    subset = df[
        df["split"] == split
    ].copy()

    print(f"\n{split.upper()}")
    print("-" * 70)

    print(
        f"Images in manifest : {len(subset):,}"
    )

    # Sample up to 10 images
    sample_n = min(
        10,
        len(subset)
    )

    sample = subset.sample(
        n=sample_n,
        random_state=42
    )

    split_samples[split] = sample

    valid_pairs = 0
    invalid_pairs = 0
    empty_labels = 0

    for _, row in sample.iterrows():

        image_path = Path(
            row["image_path"]
        )

        image_stem = Path(
            row["image_filename"]
        ).stem

        label_path = (
            LABEL_ROOT
            / split
            / f"{image_stem}.txt"
        )

        image_ok = image_path.exists()
        label_ok = label_path.exists()

        if not image_ok or not label_ok:

            invalid_pairs += 1

            print(
                "\n✗ INVALID PAIR"
            )

            print(
                "Image:",
                image_path
            )

            print(
                "Label:",
                label_path
            )

            continue

        valid_pairs += 1

        # ----------------------------------------------------
        # Check label contents
        # ----------------------------------------------------

        with open(
            label_path,
            "r"
        ) as f:

            lines = [
                line.strip()
                for line in f
                if line.strip()
            ]

        if len(lines) == 0:

            empty_labels += 1

            continue

        # ----------------------------------------------------
        # Validate YOLO rows
        # ----------------------------------------------------

        for line in lines:

            values = line.split()

            if len(values) != 5:

                raise ValueError(
                    f"Invalid YOLO row:\n"
                    f"{line}\n"
                    f"File: {label_path}"
                )

            class_id = int(values[0])

            coords = [
                float(v)
                for v in values[1:]
            ]

            assert 0 <= class_id < 15

            for value in coords:

                assert (
                    0.0 <= value <= 1.0
                )

    print(
        f"Sampled pairs       : {sample_n}"
    )

    print(
        f"Valid pairs         : {valid_pairs}"
    )

    print(
        f"Invalid pairs       : {invalid_pairs}"
    )

    print(
        f"Empty labels        : {empty_labels}"
    )

    assert invalid_pairs == 0

# ------------------------------------------------------------
# Check all image paths
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FULL CANONICAL IMAGE PATH CHECK")
print("=" * 70)

missing_images = []

for path in df["image_path"]:

    if not Path(path).exists():

        missing_images.append(
            path
        )

print(
    f"Images found   : "
    f"{len(df) - len(missing_images):,} / "
    f"{len(df):,}"
)

print(
    f"Images missing : "
    f"{len(missing_images):,}"
)

assert len(missing_images) == 0

# ------------------------------------------------------------
# Check all label paths
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FULL YOLO LABEL PATH CHECK")
print("=" * 70)

missing_labels = []

for _, row in df.iterrows():

    split = row["split"]

    stem = Path(
        row["image_filename"]
    ).stem

    label_path = (
        LABEL_ROOT
        / split
        / f"{stem}.txt"
    )

    if not label_path.exists():

        missing_labels.append(
            (
                split,
                row["image_filename"],
                str(label_path)
            )
        )

print(
    f"Labels found   : "
    f"{len(df) - len(missing_labels):,} / "
    f"{len(df):,}"
)

print(
    f"Labels missing : "
    f"{len(missing_labels):,}"
)

assert len(missing_labels) == 0

# ------------------------------------------------------------
# Verify no image-copy directories were created
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ZERO-COPY VERIFICATION")
print("=" * 70)

for directory in [
    YOLO_ROOT / "train_images",
    YOLO_ROOT / "val_images",
    YOLO_ROOT / "test_images"
]:

    if directory.exists():

        raise RuntimeError(
            f"Unexpected image-copy directory exists:\n"
            f"{directory}"
        )

print(
    "✓ No train/val/test image copies detected"
)

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ ZERO-COPY YOLO PREFLIGHT PASSED")
print("=" * 70)

print(
    "\nCanonical images remain in extracted/"
)

print(
    "Derived YOLO labels remain in yolo_baseline/labels/"
)

print(
    "No additional image storage has been created."
)

ZERO-COPY YOLO DATASET PREFLIGHT

DATASET
----------------------------------------------------------------------
Total images : 6,656
Classes      : 15

TRAIN
----------------------------------------------------------------------
Images in manifest : 5,326
Sampled pairs       : 10
Valid pairs         : 10
Invalid pairs       : 0
Empty labels        : 0

VAL
----------------------------------------------------------------------
Images in manifest : 668
Sampled pairs       : 10
Valid pairs         : 10
Invalid pairs       : 0
Empty labels        : 0

TEST
----------------------------------------------------------------------
Images in manifest : 662
Sampled pairs       : 10
Valid pairs         : 10
Invalid pairs       : 0
Empty labels        : 0

FULL CANONICAL IMAGE PATH CHECK
Images found   : 6,656 / 6,656
Images missing : 0

FULL YOLO LABEL PATH CHECK
Labels found   : 6,656 / 6,656
Labels missing : 0

ZERO-COPY VERIFICATION
✓ No train/val/test image copies detected

✓ ZERO-COPY YOLO P

In [12]:
# Create Temporary YOLO Training Input

from pathlib import Path
import pandas as pd
import os
import shutil

print("=" * 70)
print("CREATING TEMPORARY YOLO TRAINING INPUT")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SPLIT_MANIFEST = (
    PROJECT_ROOT
    / "01_data"
    / "splits"
    / "split_manifest.csv"
)

YOLO_LABEL_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
    / "labels"
)

# Temporary local dataset
TEMP_YOLO_ROOT = Path(
    "/content/yolo_baseline_runtime"
)

# ------------------------------------------------------------
# Clean previous temporary staging
# ------------------------------------------------------------

if TEMP_YOLO_ROOT.exists():

    print(
        "\nRemoving previous temporary staging..."
    )

    shutil.rmtree(
        TEMP_YOLO_ROOT
    )

# ------------------------------------------------------------
# Create structure
# ------------------------------------------------------------

for split in ["train", "val", "test"]:

    (
        TEMP_YOLO_ROOT
        / "images"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        TEMP_YOLO_ROOT
        / "labels"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "\nTemporary root:"
)
print(TEMP_YOLO_ROOT)

# ------------------------------------------------------------
# Load manifest
# ------------------------------------------------------------

df = pd.read_csv(
    SPLIT_MANIFEST
)

required = [
    "image_path",
    "image_filename",
    "split"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:

    raise ValueError(
        f"Missing manifest columns: {missing}"
    )

# ------------------------------------------------------------
# Create image symlinks
# ------------------------------------------------------------

print("\nCREATING IMAGE LINKS")
print("-" * 70)

link_counts = {}

for split in ["train", "val", "test"]:

    subset = df[
        df["split"] == split
    ]

    image_dir = (
        TEMP_YOLO_ROOT
        / "images"
        / split
    )

    label_dir = (
        TEMP_YOLO_ROOT
        / "labels"
        / split
    )

    count = 0

    for _, row in subset.iterrows():

        source_image = Path(
            row["image_path"]
        )

        filename = Path(
            row["image_filename"]
        ).name

        image_link = (
            image_dir
            / filename
        )

        label_source = (
            YOLO_LABEL_ROOT
            / split
            / f"{Path(filename).stem}.txt"
        )

        label_target = (
            label_dir
            / f"{Path(filename).stem}.txt"
        )

        # ----------------------------------------------------
        # Verify source files
        # ----------------------------------------------------

        if not source_image.exists():

            raise FileNotFoundError(
                f"Source image missing:\n"
                f"{source_image}"
            )

        if not label_source.exists():

            raise FileNotFoundError(
                f"YOLO label missing:\n"
                f"{label_source}"
            )

        # ----------------------------------------------------
        # Create symbolic link to image
        # ----------------------------------------------------

        os.symlink(
            source_image,
            image_link
        )

        # ----------------------------------------------------
        # Copy only tiny label file
        # ----------------------------------------------------

        shutil.copy2(
            label_source,
            label_target
        )

        count += 1

    link_counts[split] = count

    print(
        f"{split:<6}: "
        f"{count:,} image links + "
        f"{count:,} labels"
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TEMPORARY DATASET VERIFICATION")
print("=" * 70)

for split in ["train", "val", "test"]:

    image_dir = (
        TEMP_YOLO_ROOT
        / "images"
        / split
    )

    label_dir = (
        TEMP_YOLO_ROOT
        / "labels"
        / split
    )

    images = list(
        image_dir.iterdir()
    )

    labels = list(
        label_dir.glob("*.txt")
    )

    print(
        f"{split:<6}: "
        f"{len(images):,} images | "
        f"{len(labels):,} labels"
    )

    assert (
        len(images)
        == len(labels)
        == link_counts[split]
    )

# ------------------------------------------------------------
# Confirm images are actually symlinks
# ------------------------------------------------------------

print("\nIMAGE LINK CHECK")
print("-" * 70)

sample_image = next(
    (
        TEMP_YOLO_ROOT
        / "images"
        / "train"
    ).iterdir()
)

print(
    "Sample:",
    sample_image.name
)

print(
    "Is symbolic link:",
    sample_image.is_symlink()
)

print(
    "Points to:",
    sample_image.resolve()
)

assert sample_image.is_symlink()

# ------------------------------------------------------------
# Disk usage
# ------------------------------------------------------------

print("\nSTORAGE CHECK")
print("-" * 70)

total_size = 0

for path in (
    TEMP_YOLO_ROOT
    / "labels"
).rglob("*"):

    if path.is_file():

        total_size += path.stat().st_size

print(
    f"Local label storage : "
    f"{total_size / (1024**2):.2f} MB"
)

print(
    "Image data copied    : 0 bytes"
)

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ TEMPORARY ZERO-COPY YOLO DATASET READY")
print("=" * 70)

print(
    "\nImages remain in Google Drive."
)

print(
    "Colab uses symbolic links to the canonical images."
)

print(
    "Only lightweight YOLO label files are staged locally."
)

print(
    f"\nRuntime dataset:"
)
print(TEMP_YOLO_ROOT)

CREATING TEMPORARY YOLO TRAINING INPUT

Temporary root:
/content/yolo_baseline_runtime

CREATING IMAGE LINKS
----------------------------------------------------------------------
train : 5,326 image links + 5,326 labels
val   : 668 image links + 668 labels
test  : 662 image links + 662 labels

TEMPORARY DATASET VERIFICATION
train : 5,326 images | 5,326 labels
val   : 668 images | 668 labels
test  : 662 images | 662 labels

IMAGE LINK CHECK
----------------------------------------------------------------------
Sample: 11089kk22wcd432619_108.jpeg
Is symbolic link: True
Points to: /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Clicks/intel Real Sense Depth_Clicks/11089kk22wcd432619_108.jpeg

STORAGE CHECK
----------------------------------------------------------------------
Local label storage : 3.20 MB
Image data copied    : 0 bytes

✓ TEMPORARY ZERO-COPY YOLO DATASET READY

Images remain in Google Drive.
Colab uses symbolic 

In [13]:
# ============================================================
# Cell 23: Create YOLO Dataset Configuration
# ============================================================

from pathlib import Path
import yaml

print("=" * 70)
print("CREATING YOLO DATASET CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

YOLO_RUNTIME_ROOT = Path(
    "/content/yolo_baseline_runtime"
)

DATA_YAML = (
    YOLO_RUNTIME_ROOT
    / "data.yaml"
)

# ------------------------------------------------------------
# Class definitions
# ------------------------------------------------------------

# IMPORTANT:
# These are the numerical class IDs already present in the
# dataset. Do not rename/reorder them without changing the
# dataset itself.

CLASS_NAMES = {
    0: "class_0",
    1: "class_1",
    2: "class_2",
    3: "class_3",
    4: "class_4",
    5: "class_5",
    6: "class_6",
    7: "class_7",
    8: "class_8",
    9: "class_9",
    10: "class_10",
    11: "class_11",
    12: "class_12",
    13: "class_13",
    14: "class_14",
}

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

data_config = {
    "path": str(YOLO_RUNTIME_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 15,
    "names": CLASS_NAMES,
}

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

with open(
    DATA_YAML,
    "w"
) as f:

    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False
    )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nYOLO DATASET CONFIGURATION")
print("-" * 70)

print(
    f"Root   : {YOLO_RUNTIME_ROOT}"
)

print(
    f"Train  : images/train"
)

print(
    f"Val    : images/val"
)

print(
    f"Test   : images/test"
)

print(
    f"Classes: 15"
)

print(
    f"\nSaved:"
)

print(DATA_YAML)

# ------------------------------------------------------------
# Reload and verify
# ------------------------------------------------------------

with open(
    DATA_YAML,
    "r"
) as f:

    verified_config = yaml.safe_load(f)

assert verified_config["nc"] == 15
assert verified_config["train"] == "images/train"
assert verified_config["val"] == "images/val"
assert verified_config["test"] == "images/test"

assert len(
    verified_config["names"]
) == 15

print("\n" + "=" * 70)
print("✓ YOLO DATA.YAML CREATED AND VERIFIED")
print("=" * 70)

CREATING YOLO DATASET CONFIGURATION

YOLO DATASET CONFIGURATION
----------------------------------------------------------------------
Root   : /content/yolo_baseline_runtime
Train  : images/train
Val    : images/val
Test   : images/test
Classes: 15

Saved:
/content/yolo_baseline_runtime/data.yaml

✓ YOLO DATA.YAML CREATED AND VERIFIED


In [14]:
# ============================================================
# Cell 24: Ultralytics YOLO Dataset Loader Preflight
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import yaml
import os

print("=" * 70)
print("ULTRALYTICS YOLO DATASET LOADER PREFLIGHT")
print("=" * 70)

DATA_YAML = (
    Path("/content/yolo_baseline_runtime")
    / "data.yaml"
)

# ------------------------------------------------------------
# Verify configuration
# ------------------------------------------------------------

assert DATA_YAML.exists(), (
    f"data.yaml not found: {DATA_YAML}"
)

with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

print("\nDATA CONFIGURATION")
print("-" * 70)

print(f"Dataset root : {data_config['path']}")
print(f"Train        : {data_config['train']}")
print(f"Val          : {data_config['val']}")
print(f"Test         : {data_config['test']}")
print(f"Classes      : {data_config['nc']}")

assert data_config["nc"] == 15

# ------------------------------------------------------------
# Verify runtime directories
# ------------------------------------------------------------

runtime_root = Path(
    data_config["path"]
)

for split in ["train", "val", "test"]:

    image_dir = (
        runtime_root
        / "images"
        / split
    )

    label_dir = (
        runtime_root
        / "labels"
        / split
    )

    assert image_dir.exists(), (
        f"Missing image directory: {image_dir}"
    )

    assert label_dir.exists(), (
        f"Missing label directory: {label_dir}"
    )

# ------------------------------------------------------------
# Verify image/label counts
# ------------------------------------------------------------

print("\nRUNTIME DATASET")
print("-" * 70)

expected = {
    "train": 5326,
    "val": 668,
    "test": 662,
}

for split in ["train", "val", "test"]:

    image_dir = (
        runtime_root
        / "images"
        / split
    )

    label_dir = (
        runtime_root
        / "labels"
        / split
    )

    images = [
        p for p in image_dir.iterdir()
        if p.is_file()
    ]

    labels = list(
        label_dir.glob("*.txt")
    )

    print(
        f"{split:<6}: "
        f"{len(images):,} images | "
        f"{len(labels):,} labels"
    )

    assert len(images) == expected[split]
    assert len(labels) == expected[split]

# ------------------------------------------------------------
# Load YOLO model
# ------------------------------------------------------------

print("\nLOADING YOLO MODEL")
print("-" * 70)

model = YOLO("yolo11n.pt")

print(
    "Model loaded successfully:"
)
print(
    f"Model: {model.model.__class__.__name__}"
)

# ------------------------------------------------------------
# Perform a minimal validation pass
# ------------------------------------------------------------

print("\nRUNNING DATASET LOADER TEST")
print("-" * 70)

try:

    validation_results = model.val(
        data=str(DATA_YAML),
        imgsz=640,
        batch=4,
        device=0,
        workers=2,
        plots=False,
        verbose=False,
        split="val",
    )

    print(
        "\n✓ Ultralytics successfully loaded "
        "the validation dataset."
    )

except Exception as e:

    print(
        "\n✗ DATASET LOADER TEST FAILED"
    )

    print(
        f"\n{type(e).__name__}: {e}"
    )

    raise

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ ULTRALYTICS DATASET LOADER PREFLIGHT PASSED")
print("=" * 70)

print(
    "\nThe dataset is ready for baseline training."
)

print(
    "No image copies were created in Google Drive."
)

print(
    "GPU device requested: 0"
)

ULTRALYTICS YOLO DATASET LOADER PREFLIGHT

DATA CONFIGURATION
----------------------------------------------------------------------
Dataset root : /content/yolo_baseline_runtime
Train        : images/train
Val          : images/val
Test         : images/test
Classes      : 15

RUNTIME DATASET
----------------------------------------------------------------------
train : 5,326 images | 5,326 labels
val   : 668 images | 668 labels
test  : 662 images | 662 labels

LOADING YOLO MODEL
----------------------------------------------------------------------
Model loaded successfully:
Model: DetectionModel

RUNNING DATASET LOADER TEST
----------------------------------------------------------------------
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.2 ms, read: 0.8±0.1 MB/s, size: 512.8 KB). Use local storage instead of

In [15]:
# ============================================================
# Cell 25: Check Local Colab Storage Before Training Staging
# ============================================================

import shutil
from pathlib import Path

print("=" * 70)
print("LOCAL COLAB STORAGE CHECK")
print("=" * 70)

total, used, free = shutil.disk_usage("/content")

print("\n/content STORAGE")
print("-" * 70)

print(
    f"Total : {total / (1024**3):.2f} GB"
)

print(
    f"Used  : {used / (1024**3):.2f} GB"
)

print(
    f"Free  : {free / (1024**3):.2f} GB"
)

# ------------------------------------------------------------
# Estimate image dataset size
# ------------------------------------------------------------

manifest = pd.read_csv(
    SPLIT_MANIFEST
)

image_paths = [
    Path(p)
    for p in manifest["image_path"]
]

print("\nCANONICAL IMAGE DATASET")
print("-" * 70)

total_image_bytes = 0
missing = 0

for path in image_paths:

    if path.exists():

        total_image_bytes += (
            path.stat().st_size
        )

    else:

        missing += 1

print(
    f"Images          : {len(image_paths):,}"
)

print(
    f"Missing         : {missing:,}"
)

print(
    f"Image size      : "
    f"{total_image_bytes / (1024**3):.2f} GB"
)

print(
    f"Estimated total : "
    f"{total_image_bytes / (1024**3):.2f} GB"
)

# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

required_space = (
    total_image_bytes * 1.15
)

print("\nSAFETY CHECK")
print("-" * 70)

print(
    f"Estimated required : "
    f"{required_space / (1024**3):.2f} GB"
)

print(
    f"Available          : "
    f"{free / (1024**3):.2f} GB"
)

if free > required_space:

    print(
        "\n✓ Sufficient local storage "
        "for temporary training dataset"
    )

else:

    print(
        "\n⚠ Insufficient local storage "
        "for full temporary dataset"
    )

print("\n" + "=" * 70)
print("✓ STORAGE CHECK COMPLETE")
print("=" * 70)

LOCAL COLAB STORAGE CHECK

/content STORAGE
----------------------------------------------------------------------
Total : 112.64 GB
Used  : 47.77 GB
Free  : 64.85 GB

CANONICAL IMAGE DATASET
----------------------------------------------------------------------
Images          : 6,656
Missing         : 0
Image size      : 2.76 GB
Estimated total : 2.76 GB

SAFETY CHECK
----------------------------------------------------------------------
Estimated required : 3.18 GB
Available          : 64.85 GB

✓ Sufficient local storage for temporary training dataset

✓ STORAGE CHECK COMPLETE


In [16]:
# ============================================================
# Cell 26: Stage Canonical Images to Local Colab Storage
# ============================================================

from pathlib import Path
import pandas as pd
import shutil
import os

print("=" * 70)
print("STAGING IMAGES TO LOCAL COLAB STORAGE")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

LOCAL_ROOT = Path(
    "/content/yolo_baseline_local"
)

LOCAL_IMAGE_ROOT = (
    LOCAL_ROOT / "images"
)

LOCAL_LABEL_ROOT = (
    LOCAL_ROOT / "labels"
)

DRIVE_LABEL_ROOT = (
    PROJECT_ROOT
    / "01_data"
    / "yolo_baseline"
    / "labels"
)

# ------------------------------------------------------------
# Clean previous staging
# ------------------------------------------------------------

if LOCAL_ROOT.exists():

    print(
        "\nRemoving previous local staging..."
    )

    shutil.rmtree(
        LOCAL_ROOT
    )

# ------------------------------------------------------------
# Create directories
# ------------------------------------------------------------

for split in ["train", "val", "test"]:

    (
        LOCAL_IMAGE_ROOT / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        LOCAL_LABEL_ROOT / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Load verified split manifest
# ------------------------------------------------------------

manifest = pd.read_csv(
    SPLIT_MANIFEST
)

print(
    f"\nManifest rows : {len(manifest):,}"
)

# ------------------------------------------------------------
# Stage images and labels
# ------------------------------------------------------------

print("\nCOPYING DATA")
print("-" * 70)

expected_counts = {
    "train": 5326,
    "val": 668,
    "test": 662,
}

for split in ["train", "val", "test"]:

    subset = manifest[
        manifest["split"] == split
    ]

    assert len(subset) == expected_counts[split]

    image_target = (
        LOCAL_IMAGE_ROOT / split
    )

    label_target = (
        LOCAL_LABEL_ROOT / split
    )

    copied_images = 0
    copied_labels = 0

    for _, row in subset.iterrows():

        # ----------------------------------------------------
        # Source image
        # ----------------------------------------------------

        source_image = Path(
            row["image_path"]
        )

        filename = Path(
            row["image_filename"]
        ).name

        if not source_image.exists():

            raise FileNotFoundError(
                f"Missing source image:\n"
                f"{source_image}"
            )

        target_image = (
            image_target / filename
        )

        shutil.copy2(
            source_image,
            target_image
        )

        copied_images += 1

        # ----------------------------------------------------
        # Source YOLO label
        # ----------------------------------------------------

        source_label = (
            DRIVE_LABEL_ROOT
            / split
            / f"{Path(filename).stem}.txt"
        )

        if not source_label.exists():

            raise FileNotFoundError(
                f"Missing YOLO label:\n"
                f"{source_label}"
            )

        target_label = (
            label_target
            / f"{Path(filename).stem}.txt"
        )

        shutil.copy2(
            source_label,
            target_label
        )

        copied_labels += 1

    print(
        f"{split:<6}: "
        f"{copied_images:,} images | "
        f"{copied_labels:,} labels"
    )

# ------------------------------------------------------------
# Verify local staging
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOCAL DATASET VERIFICATION")
print("=" * 70)

for split in ["train", "val", "test"]:

    image_dir = (
        LOCAL_IMAGE_ROOT / split
    )

    label_dir = (
        LOCAL_LABEL_ROOT / split
    )

    images = [
        p for p in image_dir.iterdir()
        if p.is_file()
    ]

    labels = list(
        label_dir.glob("*.txt")
    )

    print(
        f"{split:<6}: "
        f"{len(images):,} images | "
        f"{len(labels):,} labels"
    )

    assert len(images) == expected_counts[split]
    assert len(labels) == expected_counts[split]

# ------------------------------------------------------------
# Confirm images are REAL local files
# ------------------------------------------------------------

print("\nIMAGE STORAGE CHECK")
print("-" * 70)

sample = next(
    (
        LOCAL_IMAGE_ROOT / "train"
    ).iterdir()
)

print(
    f"Sample image : {sample.name}"
)

print(
    f"Is symlink   : {sample.is_symlink()}"
)

print(
    f"File size    : "
    f"{sample.stat().st_size / 1024:.1f} KB"
)

assert not sample.is_symlink()

# ------------------------------------------------------------
# Calculate local storage used
# ------------------------------------------------------------

total_bytes = 0

for path in LOCAL_ROOT.rglob("*"):

    if path.is_file():

        total_bytes += path.stat().st_size

print("\nLOCAL STORAGE USED")
print("-" * 70)

print(
    f"Total : "
    f"{total_bytes / (1024**3):.2f} GB"
)

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ LOCAL YOLO TRAINING DATASET READY")
print("=" * 70)

print(
    "\nCanonical images in Drive: PRESERVED"
)

print(
    "Local training images     : TEMPORARY"
)

print(
    "Google Drive image copy   : NONE"
)

print(
    f"\nLocal dataset:"
)

print(
    LOCAL_ROOT
)

STAGING IMAGES TO LOCAL COLAB STORAGE

Manifest rows : 6,656

COPYING DATA
----------------------------------------------------------------------
train : 5,326 images | 5,326 labels
val   : 668 images | 668 labels
test  : 662 images | 662 labels

LOCAL DATASET VERIFICATION
train : 5,326 images | 5,326 labels
val   : 668 images | 668 labels
test  : 662 images | 662 labels

IMAGE STORAGE CHECK
----------------------------------------------------------------------
Sample image : 11089kk22wcd432619_108.jpeg
Is symlink   : False
File size    : 504.5 KB

LOCAL STORAGE USED
----------------------------------------------------------------------
Total : 2.77 GB

✓ LOCAL YOLO TRAINING DATASET READY

Canonical images in Drive: PRESERVED
Local training images     : TEMPORARY
Google Drive image copy   : NONE

Local dataset:
/content/yolo_baseline_local


In [17]:
# ============================================================
# Cell 27: Create Local YOLO Training Configuration
# ============================================================

from pathlib import Path
import yaml

print("=" * 70)
print("CREATING LOCAL YOLO TRAINING CONFIGURATION")
print("=" * 70)

LOCAL_ROOT = Path("/content/yolo_baseline_local")

LOCAL_DATA_YAML = LOCAL_ROOT / "data.yaml"

# ------------------------------------------------------------
# Class mapping
# ------------------------------------------------------------

CLASS_NAMES = [
    "class_0",
    "class_1",
    "class_2",
    "class_3",
    "class_4",
    "class_5",
    "class_6",
    "class_7",
    "class_8",
    "class_9",
    "class_10",
    "class_11",
    "class_12",
    "class_13",
    "class_14",
]

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

local_config = {
    "path": str(LOCAL_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 15,
    "names": CLASS_NAMES,
}

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

with open(LOCAL_DATA_YAML, "w") as f:
    yaml.safe_dump(
        local_config,
        f,
        sort_keys=False
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

with open(LOCAL_DATA_YAML, "r") as f:
    verified = yaml.safe_load(f)

assert verified["path"] == str(LOCAL_ROOT)
assert verified["train"] == "images/train"
assert verified["val"] == "images/val"
assert verified["test"] == "images/test"
assert verified["nc"] == 15
assert len(verified["names"]) == 15

print("\nLOCAL DATASET")
print("-" * 70)
print(f"Root    : {LOCAL_ROOT}")
print("Train   : images/train")
print("Val     : images/val")
print("Test    : images/test")
print("Classes : 15")

print("\nSaved:")
print(LOCAL_DATA_YAML)

print("\n" + "=" * 70)
print("✓ LOCAL TRAINING DATA.YAML CREATED AND VERIFIED")
print("=" * 70)

CREATING LOCAL YOLO TRAINING CONFIGURATION

LOCAL DATASET
----------------------------------------------------------------------
Root    : /content/yolo_baseline_local
Train   : images/train
Val     : images/val
Test    : images/test
Classes : 15

Saved:
/content/yolo_baseline_local/data.yaml

✓ LOCAL TRAINING DATA.YAML CREATED AND VERIFIED


In [18]:
# ============================================================
# Cell 28: Baseline YOLO Training Dry Run
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch
import json
import time

print("=" * 70)
print("BASELINE YOLO TRAINING DRY RUN")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

LOCAL_ROOT = Path("/content/yolo_baseline_local")

LOCAL_DATA_YAML = (
    LOCAL_ROOT / "data.yaml"
)

DRY_RUN_DIR = (
    Path(EXPERIMENT_RESULTS)
    / "dry_run"
)

DRY_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

print("\nENVIRONMENT")
print("-" * 70)

print(
    f"PyTorch : {torch.__version__}"
)

print(
    f"CUDA    : {torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"GPU     : {torch.cuda.get_device_name(0)}"
    )

else:

    raise RuntimeError(
        "CUDA GPU is not available. "
        "Do NOT continue to full training."
    )

# ------------------------------------------------------------
# Load pretrained model
# ------------------------------------------------------------

print("\nMODEL")
print("-" * 70)

model = YOLO("yolo11n.pt")

print(
    "✓ yolo11n.pt loaded"
)

# ------------------------------------------------------------
# Dry run
# ------------------------------------------------------------

print("\nRUNNING 1-EPOCH DRY RUN")
print("-" * 70)

start_time = time.time()

dry_results = model.train(

    data=str(LOCAL_DATA_YAML),

    # One epoch only
    epochs=1,

    # Use a small subset for pipeline testing
    fraction=0.02,

    imgsz=640,

    batch=16,

    device=0,

    workers=2,

    seed=42,

    deterministic=True,

    # Do not spend time on extensive plotting
    plots=False,

    cache=False,

    # Disable long early stopping logic
    patience=1,

    # Save checkpoint
    save=True,

    project=str(DRY_RUN_DIR),

    name="pipeline_test",

    exist_ok=True,

    verbose=True,
)

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DRY RUN COMPLETE")
print("=" * 70)

print(
    f"\nRuntime : {elapsed / 60:.2f} minutes"
)

print(
    f"Output  : {DRY_RUN_DIR / 'pipeline_test'}"
)

# ------------------------------------------------------------
# Verify output directory
# ------------------------------------------------------------

run_dir = (
    DRY_RUN_DIR / "pipeline_test"
)

print("\nOUTPUT VERIFICATION")
print("-" * 70)

if run_dir.exists():

    print(
        "✓ Training output directory exists"
    )

else:

    raise FileNotFoundError(
        f"Training output directory missing: {run_dir}"
    )

weights_dir = (
    run_dir / "weights"
)

if weights_dir.exists():

    print(
        "✓ Weights directory exists"
    )

else:

    raise FileNotFoundError(
        f"Weights directory missing: {weights_dir}"
    )

best_pt = (
    weights_dir / "best.pt"
)

last_pt = (
    weights_dir / "last.pt"
)

print(
    f"best.pt : {'✓' if best_pt.exists() else '✗'}"
)

print(
    f"last.pt : {'✓' if last_pt.exists() else '✗'}"
)

# ------------------------------------------------------------
# Save dry-run metadata
# ------------------------------------------------------------

metadata = {
    "experiment": EXPERIMENT_ID,
    "model": "yolo11n.pt",
    "epochs": 1,
    "fraction": 0.02,
    "image_size": 640,
    "batch": 16,
    "device": 0,
    "gpu": torch.cuda.get_device_name(0),
    "pytorch": torch.__version__,
    "elapsed_seconds": elapsed,
    "dataset": str(LOCAL_DATA_YAML),
}

metadata_path = (
    run_dir / "dry_run_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

print(
    f"\nMetadata saved:"
)

print(
    metadata_path
)

print("\n" + "=" * 70)
print("✓ TRAINING PIPELINE DRY RUN PASSED")
print("=" * 70)

print(
    "\nThe full baseline experiment can now be started."
)

BASELINE YOLO TRAINING DRY RUN

ENVIRONMENT
----------------------------------------------------------------------
PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : Tesla T4

MODEL
----------------------------------------------------------------------
✓ yolo11n.pt loaded

RUNNING 1-EPOCH DRY RUN
----------------------------------------------------------------------
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_baseline_local/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasi

In [19]:
# ============================================================
# Cell 29: Final Baseline YOLO Training Configuration
# ============================================================

from pathlib import Path
import json
import torch
import ultralytics

print("=" * 70)
print("FINAL BASELINE YOLO TRAINING CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

LOCAL_ROOT = Path(
    "/content/yolo_baseline_local"
)

LOCAL_DATA_YAML = (
    LOCAL_ROOT / "data.yaml"
)

RESULTS_DIR = Path(
    EXPERIMENT_RESULTS
)

CHECKPOINT_DIR = Path(
    EXPERIMENT_CHECKPOINTS
)

LOG_DIR = Path(
    EXPERIMENT_LOGS
)

CONFIG_DIR = (
    PROJECT_ROOT / "configs"
)

# ------------------------------------------------------------
# Verify persistent experiment directories
# ------------------------------------------------------------

for directory in [
    RESULTS_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
    CONFIG_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

assert LOCAL_DATA_YAML.exists()

# ------------------------------------------------------------
# Final training configuration
# ------------------------------------------------------------

training_config = {

    "experiment_id": EXPERIMENT_ID,

    "task": "detect",

    "model": "yolo11n.pt",

    "pretrained": True,

    "dataset": {
        "yaml": str(LOCAL_DATA_YAML),
        "train_images": 5326,
        "val_images": 668,
        "test_images": 662,
        "classes": 15,
        "valid_boxes": 62040,
        "excluded_invalid_boxes": 12,
        "group_leakage": False,
    },

    "training": {
        "epochs": 100,
        "image_size": 640,
        "batch": 16,
        "device": 0,
        "workers": 2,
        "seed": 42,
        "deterministic": True,
        "patience": 20,
        "save": True,
        "plots": True,
        "cache": False,
        "fraction": 1.0,
    },

    "environment": {
        "python": "3.13.15",
        "ultralytics": ultralytics.__version__,
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_memory_gb": round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
    },

    "storage": {
        "canonical_images": "Google Drive",
        "training_images": str(LOCAL_ROOT),
        "training_image_storage": "temporary_colab_local",
        "checkpoints": str(CHECKPOINT_DIR),
        "logs": str(LOG_DIR),
        "results": str(RESULTS_DIR),
    },

    "reproducibility": {
        "random_seed": 42,
        "split_seed": 42,
        "split_strategy": "leakage-aware group split",
        "train_groups": 5151,
        "val_groups": 644,
        "test_groups": 644,
    },

}

# ------------------------------------------------------------
# Save experiment configuration
# ------------------------------------------------------------

experiment_config_path = (
    RESULTS_DIR /
    "final_baseline_training_config.json"
)

with open(
    experiment_config_path,
    "w"
) as f:

    json.dump(
        training_config,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Save central copy
# ------------------------------------------------------------

central_config_path = (
    CONFIG_DIR /
    f"{EXPERIMENT_ID}_final_training.json"
)

with open(
    central_config_path,
    "w"
) as f:

    json.dump(
        training_config,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Print configuration
# ------------------------------------------------------------

print("\nMODEL")
print("-" * 70)
print("Model       : yolo11n.pt")
print("Pretrained  : True")
print("Task        : Detection")
print("Classes     : 15")

print("\nDATASET")
print("-" * 70)
print("Train images : 5,326")
print("Val images   : 668")
print("Test images  : 662")
print("Valid boxes  : 62,040")
print("Group leak   : False")

print("\nTRAINING")
print("-" * 70)
print("Epochs       : 100")
print("Image size   : 640")
print("Batch        : 16")
print("Device       : 0")
print("Workers      : 2")
print("Seed         : 42")
print("Deterministic: True")
print("Patience     : 20")
print("Cache        : False")
print("Fraction     : 1.0")

print("\nENVIRONMENT")
print("-" * 70)
print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.version.cuda}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")

print("\nSAVED CONFIGS")
print("-" * 70)
print(experiment_config_path)
print(central_config_path)

print("\n" + "=" * 70)
print("✓ FINAL BASELINE CONFIGURATION LOCKED")
print("=" * 70)

FINAL BASELINE YOLO TRAINING CONFIGURATION

MODEL
----------------------------------------------------------------------
Model       : yolo11n.pt
Pretrained  : True
Task        : Detection
Classes     : 15

DATASET
----------------------------------------------------------------------
Train images : 5,326
Val images   : 668
Test images  : 662
Valid boxes  : 62,040
Group leak   : False

TRAINING
----------------------------------------------------------------------
Epochs       : 100
Image size   : 640
Batch        : 16
Device       : 0
Workers      : 2
Seed         : 42
Deterministic: True
Patience     : 20
Cache        : False
Fraction     : 1.0

ENVIRONMENT
----------------------------------------------------------------------
Ultralytics : 8.4.144
PyTorch     : 2.11.0+cu128
CUDA        : 12.8
GPU         : Tesla T4

SAVED CONFIGS
----------------------------------------------------------------------
/content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_09

In [20]:
# ============================================================
# Cell 30: FULL BASELINE YOLO TRAINING
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch
import time
import json

print("=" * 70)
print("STARTING FULL BASELINE YOLO TRAINING")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

LOCAL_ROOT = Path(
    "/content/yolo_baseline_local"
)

DATA_YAML = LOCAL_ROOT / "data.yaml"

RESULTS_DIR = Path(EXPERIMENT_RESULTS)
CHECKPOINT_DIR = Path(EXPERIMENT_CHECKPOINTS)
LOG_DIR = Path(EXPERIMENT_LOGS)

RUN_NAME = "baseline_training"

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert DATA_YAML.exists(), (
    f"Missing dataset configuration: {DATA_YAML}"
)

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Stop before training."
)

gpu_name = torch.cuda.get_device_name(0)

print("\nTRAINING ENVIRONMENT")
print("-" * 70)
print(f"GPU          : {gpu_name}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.version.cuda}")
print(f"Dataset YAML : {DATA_YAML}")

# ------------------------------------------------------------
# Load pretrained model
# ------------------------------------------------------------

print("\nLOADING MODEL")
print("-" * 70)

model = YOLO("yolo11n.pt")

print("✓ YOLO11n pretrained model loaded")

# ------------------------------------------------------------
# Start training
# ------------------------------------------------------------

print("\nTRAINING CONFIGURATION")
print("-" * 70)
print("Epochs        : 100")
print("Image size    : 640")
print("Batch         : 16")
print("Device        : 0")
print("Workers       : 2")
print("Seed          : 42")
print("Deterministic : True")
print("Patience      : 20")
print("Cache         : False")
print("Fraction      : 1.0")
print("Plots         : True")
print("Save          : True")

print("\n" + "=" * 70)
print("TRAINING STARTING")
print("=" * 70)

start_time = time.time()

results = model.train(

    data=str(DATA_YAML),

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    epochs=100,
    imgsz=640,
    batch=16,

    # --------------------------------------------------------
    # Hardware
    # --------------------------------------------------------

    device=0,
    workers=2,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=42,
    deterministic=True,

    # --------------------------------------------------------
    # Training control
    # --------------------------------------------------------

    patience=20,

    # --------------------------------------------------------
    # Storage / performance
    # --------------------------------------------------------

    cache=False,
    save=True,
    plots=True,

    # --------------------------------------------------------
    # Full dataset
    # --------------------------------------------------------

    fraction=1.0,

    # --------------------------------------------------------
    # Persistent experiment output
    # --------------------------------------------------------

    project=str(RESULTS_DIR),
    name=RUN_NAME,
    exist_ok=True,

    verbose=True,
)

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Training complete
# ------------------------------------------------------------

run_dir = RESULTS_DIR / RUN_NAME

print("\n" + "=" * 70)
print("BASELINE TRAINING COMPLETE")
print("=" * 70)

print(
    f"\nTraining time : {elapsed / 3600:.2f} hours"
)

print(
    f"Run directory : {run_dir}"
)

# ------------------------------------------------------------
# Checkpoint verification
# ------------------------------------------------------------

weights_dir = run_dir / "weights"

best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"

print("\nCHECKPOINTS")
print("-" * 70)

print(
    f"best.pt : {'✓ EXISTS' if best_pt.exists() else '✗ MISSING'}"
)

print(
    f"last.pt : {'✓ EXISTS' if last_pt.exists() else '✗ MISSING'}"
)

# ------------------------------------------------------------
# Save training completion metadata
# ------------------------------------------------------------

metadata = {
    "experiment_id": EXPERIMENT_ID,
    "run_name": RUN_NAME,
    "model": "yolo11n.pt",
    "epochs_requested": 100,
    "image_size": 640,
    "batch": 16,
    "device": 0,
    "gpu": gpu_name,
    "seed": 42,
    "deterministic": True,
    "patience": 20,
    "cache": False,
    "fraction": 1.0,
    "dataset_yaml": str(DATA_YAML),
    "training_time_seconds": elapsed,
    "best_checkpoint": str(best_pt),
    "last_checkpoint": str(last_pt),
}

metadata_path = (
    run_dir / "training_completion_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print(
    f"\nCompletion metadata:"
)

print(
    metadata_path
)

print("\n" + "=" * 70)

if best_pt.exists() and last_pt.exists():

    print("✓ BASELINE TRAINING COMPLETED SUCCESSFULLY")

else:

    print("⚠ TRAINING FINISHED BUT CHECKPOINT VERIFICATION FAILED")

print("=" * 70)

STARTING FULL BASELINE YOLO TRAINING

TRAINING ENVIRONMENT
----------------------------------------------------------------------
GPU          : Tesla T4
PyTorch      : 2.11.0+cu128
CUDA         : 12.8
Dataset YAML : /content/yolo_baseline_local/data.yaml

LOADING MODEL
----------------------------------------------------------------------
✓ YOLO11n pretrained model loaded

TRAINING CONFIGURATION
----------------------------------------------------------------------
Epochs        : 100
Image size    : 640
Batch         : 16
Device        : 0
Workers       : 2
Seed          : 42
Deterministic : True
Patience      : 20
Cache         : False
Fraction      : 1.0
Plots         : True
Save          : True

TRAINING STARTING
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close

In [21]:
# ============================================================
# Cell 31: HELD-OUT TEST SET EVALUATION
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import json
import time
import torch

print("=" * 70)
print("HELD-OUT TEST SET EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

LOCAL_ROOT = Path(
    "/content/yolo_baseline_local"
)

DATA_YAML = LOCAL_ROOT / "data.yaml"

RUN_DIR = (
    Path(EXPERIMENT_RESULTS)
    / "baseline_training"
)

BEST_MODEL = (
    RUN_DIR / "weights" / "best.pt"
)

TEST_RESULTS_DIR = (
    Path(EXPERIMENT_RESULTS)
    / "test_evaluation"
)

TEST_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert DATA_YAML.exists(), (
    f"Dataset configuration missing: {DATA_YAML}"
)

assert BEST_MODEL.exists(), (
    f"Best checkpoint missing: {BEST_MODEL}"
)

assert torch.cuda.is_available(), (
    "CUDA is unavailable."
)

print("\nINPUTS")
print("-" * 70)
print(f"Model       : {BEST_MODEL}")
print(f"Dataset     : {DATA_YAML}")
print(f"Test images : 662")
print(f"GPU         : {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------
# Load best checkpoint
# ------------------------------------------------------------

print("\nLOADING BEST CHECKPOINT")
print("-" * 70)

model = YOLO(str(BEST_MODEL))

print("✓ best.pt loaded successfully")

# ------------------------------------------------------------
# Evaluate ONLY on test set
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNNING HELD-OUT TEST EVALUATION")
print("=" * 70)

start_time = time.time()

test_metrics = model.val(
    data=str(DATA_YAML),

    split="test",

    imgsz=640,
    batch=16,

    device=0,
    workers=2,

    # Reproducibility
    seed=42,

    # Do not cache images
    cache=False,

    # Save evaluation artifacts
    plots=True,

    project=str(TEST_RESULTS_DIR),
    name="baseline_test",
    exist_ok=True,

    verbose=True,
)

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Extract metrics
# ------------------------------------------------------------

box_metrics = test_metrics.box

precision = float(box_metrics.mp)
recall = float(box_metrics.mr)
map50 = float(box_metrics.map50)
map50_95 = float(box_metrics.map)

# ------------------------------------------------------------
# Print official test results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OFFICIAL BASELINE TEST RESULTS")
print("=" * 70)

print("\nOVERALL")
print("-" * 70)

print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"mAP@50          : {map50:.4f}")
print(f"mAP@50-95       : {map50_95:.4f}")

print(
    f"\nEvaluation time : {elapsed / 60:.2f} minutes"
)

# ------------------------------------------------------------
# Per-class metrics
# ------------------------------------------------------------

print("\nPER-CLASS RESULTS")
print("-" * 70)

class_results = {}

if hasattr(box_metrics, "maps"):

    class_maps = box_metrics.maps

    for class_id, class_map in enumerate(class_maps):

        class_results[str(class_id)] = {
            "mAP50-95": float(class_map)
        }

        print(
            f"Class {class_id:2d} : "
            f"mAP50-95 = {float(class_map):.4f}"
        )

# ------------------------------------------------------------
# Save official test metrics
# ------------------------------------------------------------

official_results = {

    "experiment_id": EXPERIMENT_ID,

    "evaluation": "held_out_test",

    "model": "yolo11n.pt",

    "checkpoint": str(BEST_MODEL),

    "dataset": {
        "test_images": 662,
        "classes": 15,
        "valid_boxes": 6154,
        "group_leakage": False,
    },

    "metrics": {
        "precision": precision,
        "recall": recall,
        "mAP50": map50,
        "mAP50_95": map50_95,
    },

    "per_class": class_results,

    "evaluation_time_seconds": elapsed,

    "configuration": {
        "imgsz": 640,
        "batch": 16,
        "device": 0,
        "seed": 42,
        "cache": False,
    },
}

metrics_path = (
    TEST_RESULTS_DIR
    / "official_baseline_test_metrics.json"
)

with open(metrics_path, "w") as f:

    json.dump(
        official_results,
        f,
        indent=2
    )

print("\nSAVED")
print("-" * 70)
print(metrics_path)

print("\n" + "=" * 70)
print("✓ HELD-OUT TEST EVALUATION COMPLETE")
print("=" * 70)

HELD-OUT TEST SET EVALUATION

INPUTS
----------------------------------------------------------------------
Model       : /content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_095854/baseline_training/weights/best.pt
Dataset     : /content/yolo_baseline_local/data.yaml
Test images : 662
GPU         : Tesla T4

LOADING BEST CHECKPOINT
----------------------------------------------------------------------
✓ best.pt loaded successfully

RUNNING HELD-OUT TEST EVALUATION
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,585,077 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 52.3±35.4 MB/s, size: 354.4 KB)
val: Scanning /content/yolo_baseline_local/labels/test... 662 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 662/662 241.1it/s 2.7s
val: New cache created: /content/yolo_baseline_local/labels/test.cache
                 Class     Images  Instances

In [22]:
# BASELINE EXPERIMENT AUDIT
from pathlib import Path
import json
import pandas as pd
import torch
import ultralytics

print("=" * 70)
print("BASELINE YOLO EXPERIMENT AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dissertation_weed_detection"
)

RESULTS_DIR = Path(EXPERIMENT_RESULTS)
RUN_DIR = RESULTS_DIR / "baseline_training"
TEST_DIR = RESULTS_DIR / "test_evaluation"

BEST_MODEL = RUN_DIR / "weights" / "best.pt"
LAST_MODEL = RUN_DIR / "weights" / "last.pt"

TEST_METRICS = (
    TEST_DIR /
    "official_baseline_test_metrics.json"
)

FINAL_CONFIG = (
    RESULTS_DIR /
    "final_baseline_training_config.json"
)

SPLIT_MANIFEST = (
    PROJECT_ROOT /
    "01_data" /
    "splits" /
    "split_manifest.csv"
)

# ------------------------------------------------------------
# Verify all critical artifacts
# ------------------------------------------------------------

artifacts = {
    "Best checkpoint": BEST_MODEL,
    "Last checkpoint": LAST_MODEL,
    "Test metrics": TEST_METRICS,
    "Final training config": FINAL_CONFIG,
    "Split manifest": SPLIT_MANIFEST,
}

print("\nCRITICAL ARTIFACTS")
print("-" * 70)

all_present = True

for name, path in artifacts.items():

    exists = path.exists()

    print(
        f"{name:<25}: "
        f"{'✓ EXISTS' if exists else '✗ MISSING'}"
    )

    if not exists:
        all_present = False

# ------------------------------------------------------------
# Load test metrics
# ------------------------------------------------------------

assert TEST_METRICS.exists()

with open(TEST_METRICS, "r") as f:
    test_data = json.load(f)

metrics = test_data["metrics"]

# ------------------------------------------------------------
# Load split manifest
# ------------------------------------------------------------

assert SPLIT_MANIFEST.exists()

split_df = pd.read_csv(SPLIT_MANIFEST)

split_counts = (
    split_df["split"]
    .value_counts()
    .to_dict()
)

# ------------------------------------------------------------
# Audit summary
# ------------------------------------------------------------

print("\nDATASET")
print("-" * 70)

print(f"Total images : {len(split_df):,}")
print(f"Train       : {split_counts.get('train', 0):,}")
print(f"Val         : {split_counts.get('val', 0):,}")
print(f"Test        : {split_counts.get('test', 0):,}")

print("\nOFFICIAL TEST PERFORMANCE")
print("-" * 70)

print(
    f"Precision  : {metrics['precision']:.4f}"
)

print(
    f"Recall     : {metrics['recall']:.4f}"
)

print(
    f"mAP@50     : {metrics['mAP50']:.4f}"
)

print(
    f"mAP@50-95  : {metrics['mAP50_95']:.4f}"
)

print("\nENVIRONMENT")
print("-" * 70)

print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.version.cuda}")

if torch.cuda.is_available():
    print(
        f"GPU         : "
        f"{torch.cuda.get_device_name(0)}"
    )

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if all_present:
    print("✓ BASELINE EXPERIMENT ARTIFACT AUDIT PASSED")
else:
    print("✗ BASELINE EXPERIMENT ARTIFACT AUDIT FAILED")

print("=" * 70)

BASELINE YOLO EXPERIMENT AUDIT

CRITICAL ARTIFACTS
----------------------------------------------------------------------
Best checkpoint          : ✓ EXISTS
Last checkpoint          : ✓ EXISTS
Test metrics             : ✓ EXISTS
Final training config    : ✓ EXISTS
Split manifest           : ✓ EXISTS

DATASET
----------------------------------------------------------------------
Total images : 6,656
Train       : 5,326
Val         : 668
Test        : 662

OFFICIAL TEST PERFORMANCE
----------------------------------------------------------------------
Precision  : 0.6706
Recall     : 0.6113
mAP@50     : 0.6392
mAP@50-95  : 0.3156

ENVIRONMENT
----------------------------------------------------------------------
Ultralytics : 8.4.144
PyTorch     : 2.11.0+cu128
CUDA        : 12.8
GPU         : Tesla T4

✓ BASELINE EXPERIMENT ARTIFACT AUDIT PASSED


In [23]:
# BASELINE RESULTS SUMMARY

print("=" * 70)
print("CREATING BASELINE RESULTS SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# Load official test metrics
# ------------------------------------------------------------

with open(TEST_METRICS, "r") as f:
    test_data = json.load(f)

m = test_data["metrics"]

# ------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------

baseline_summary = pd.DataFrame([{

    "experiment_id": EXPERIMENT_ID,

    "model": "YOLO11n",

    "pretrained": True,

    "train_images": 5326,
    "val_images": 668,
    "test_images": 662,

    "classes": 15,

    "valid_boxes": 62040,

    "excluded_invalid_boxes": 12,

    "group_leakage": False,

    "epochs": 100,
    "image_size": 640,
    "batch_size": 16,

    "precision": m["precision"],
    "recall": m["recall"],
    "mAP50": m["mAP50"],
    "mAP50_95": m["mAP50_95"],

    "gpu": "Tesla T4",

    "seed": 42,

}])

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

summary_path = (
    TEST_DIR /
    "baseline_results_summary.csv"
)

baseline_summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nBASELINE SUMMARY")
print("-" * 70)

print(
    baseline_summary.to_string(index=False)
)

print("\nSaved:")
print(summary_path)

print("\n" + "=" * 70)
print("✓ BASELINE RESULTS SUMMARY SAVED")
print("=" * 70)

CREATING BASELINE RESULTS SUMMARY

BASELINE SUMMARY
----------------------------------------------------------------------
                experiment_id   model  pretrained  train_images  val_images  test_images  classes  valid_boxes  excluded_invalid_boxes  group_leakage  epochs  image_size  batch_size  precision   recall    mAP50  mAP50_95      gpu  seed
baseline_yolo_20260909_095854 YOLO11n        True          5326         668          662       15        62040                      12          False     100         640          16   0.670641 0.611301 0.639216  0.315558 Tesla T4    42

Saved:
/content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_095854/test_evaluation/baseline_results_summary.csv

✓ BASELINE RESULTS SUMMARY SAVED


In [24]:
# PER-CLASS BASELINE RESULTS
print("=" * 70)
print("PER-CLASS BASELINE RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Extract per-class results
# ------------------------------------------------------------

class_results = test_data.get(
    "per_class",
    {}
)

rows = []

for class_id, values in class_results.items():

    rows.append({

        "class_id": int(class_id),

        "mAP50_95": values["mAP50-95"],

    })

per_class_df = (
    pd.DataFrame(rows)
    .sort_values("class_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

per_class_path = (
    TEST_DIR /
    "baseline_per_class_metrics.csv"
)

per_class_df.to_csv(
    per_class_path,
    index=False
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n")
print(
    per_class_df.to_string(index=False)
)

print("\nSaved:")
print(per_class_path)

print("\n" + "=" * 70)
print("✓ PER-CLASS BASELINE RESULTS SAVED")
print("=" * 70)

PER-CLASS BASELINE RESULTS


 class_id  mAP50_95
        0  0.406103
        1  0.233453
        2  0.296018
        3  0.599468
        4  0.378282
        5  0.289319
        6  0.353452
        7  0.329385
        8  0.272116
        9  0.197110
       10  0.429915
       11  0.164552
       12  0.259397
       13  0.081988
       14  0.442810

Saved:
/content/drive/MyDrive/dissertation_weed_detection/results/baseline_yolo_20260909_095854/test_evaluation/baseline_per_class_metrics.csv

✓ PER-CLASS BASELINE RESULTS SAVED


In [25]:
# FINAL BASELINE EXPERIMENT RECORD

print("=" * 70)
print("CREATING FINAL BASELINE EXPERIMENT RECORD")
print("=" * 70)

record = f"""
BASELINE YOLO EXPERIMENT RECORD
======================================================================

Experiment ID
----------------------------------------------------------------------
{EXPERIMENT_ID}

MODEL
----------------------------------------------------------------------
Architecture       : YOLO11n
Framework          : Ultralytics
Pretrained         : Yes
Task               : Object Detection
Number of classes  : 15

DATASET
----------------------------------------------------------------------
Total images       : 6,656
Training images    : 5,326
Validation images  : 668
Test images        : 662

Valid annotation boxes : 62,040
Excluded malformed boxes : 12

Near-duplicate groups : 6,439
Group leakage         : None

SPLIT
----------------------------------------------------------------------
Strategy            : Leakage-aware group split
Random seed         : 42
Train groups        : 5,151
Validation groups   : 644
Test groups         : 644

TRAINING
----------------------------------------------------------------------
Epochs              : 100
Image size          : 640
Batch size          : 16
Workers             : 2
Device              : CUDA:0
GPU                 : Tesla T4
Seed                : 42
Deterministic       : True
Patience            : 20
Cache               : False
Fraction            : 1.0

OFFICIAL HELD-OUT TEST RESULTS
----------------------------------------------------------------------
Precision           : {m['precision']:.4f}
Recall              : {m['recall']:.4f}
mAP@50              : {m['mAP50']:.4f}
mAP@50-95           : {m['mAP50_95']:.4f}

PRIMARY BASELINE METRIC
----------------------------------------------------------------------
mAP@50-95           : {m['mAP50_95']:.4f}

CHECKPOINT
----------------------------------------------------------------------
Best model          : {BEST_MODEL}
Last model          : {LAST_MODEL}

TEST RESULTS
----------------------------------------------------------------------
Test metrics        : {TEST_METRICS}

GENERATED
----------------------------------------------------------------------
Date                : {datetime.now().isoformat()}

======================================================================
END OF BASELINE EXPERIMENT RECORD
======================================================================
"""

record_path = (
    RESULTS_DIR /
    "baseline_experiment_record.txt"
)

with open(record_path, "w") as f:
    f.write(record)

print("\n")
print(record)

print("\nSaved:")
print(record_path)

print("\n" + "=" * 70)
print("✓ FINAL BASELINE EXPERIMENT RECORD SAVED")
print("=" * 70)

CREATING FINAL BASELINE EXPERIMENT RECORD



BASELINE YOLO EXPERIMENT RECORD

Experiment ID
----------------------------------------------------------------------
baseline_yolo_20260909_095854

MODEL
----------------------------------------------------------------------
Architecture       : YOLO11n
Framework          : Ultralytics
Pretrained         : Yes
Task               : Object Detection
Number of classes  : 15

DATASET
----------------------------------------------------------------------
Total images       : 6,656
Training images    : 5,326
Validation images  : 668
Test images        : 662

Valid annotation boxes : 62,040
Excluded malformed boxes : 12

Near-duplicate groups : 6,439
Group leakage         : None

SPLIT
----------------------------------------------------------------------
Strategy            : Leakage-aware group split
Random seed         : 42
Train groups        : 5,151
Validation groups   : 644
Test groups         : 644

TRAINING
--------------------------------

In [26]:
# FINAL NOTEBOOK 4 AUDIT
print("=" * 70)
print("FINAL NOTEBOOK 4 AUDIT")
print("=" * 70)

final_artifacts = [

    RESULTS_DIR /
    "final_baseline_training_config.json",

    RUN_DIR /
    "weights" /
    "best.pt",

    RUN_DIR /
    "weights" /
    "last.pt",

    TEST_DIR /
    "official_baseline_test_metrics.json",

    TEST_DIR /
    "baseline_results_summary.csv",

    TEST_DIR /
    "baseline_per_class_metrics.csv",

    RESULTS_DIR /
    "baseline_experiment_record.txt",

]

print("\nREQUIRED ARTIFACTS")
print("-" * 70)

missing = []

for path in final_artifacts:

    if path.exists():

        print(f"✓ {path.name}")

    else:

        print(f"✗ {path.name}")
        missing.append(path)

print("\n" + "=" * 70)

if len(missing) == 0:

    print("✓ NOTEBOOK 4 BASELINE AUDIT PASSED")
    print("✓ ALL REQUIRED EXPERIMENT ARTIFACTS PRESENT")
    print("✓ BASELINE IS READY FOR COMPARATIVE EXPERIMENTS")

else:

    print(
        f"⚠ {len(missing)} REQUIRED ARTIFACT(S) MISSING"
    )

print("=" * 70)

FINAL NOTEBOOK 4 AUDIT

REQUIRED ARTIFACTS
----------------------------------------------------------------------
✓ final_baseline_training_config.json
✓ best.pt
✓ last.pt
✓ official_baseline_test_metrics.json
✓ baseline_results_summary.csv
✓ baseline_per_class_metrics.csv
✓ baseline_experiment_record.txt

✓ NOTEBOOK 4 BASELINE AUDIT PASSED
✓ ALL REQUIRED EXPERIMENT ARTIFACTS PRESENT
✓ BASELINE IS READY FOR COMPARATIVE EXPERIMENTS
